# LSTM-based Reduced-Time Recursive Physics-Informed Neural Network for Shape Memory Materials

## Non-Isothermal Shape-Memory Inverse Characterization 

This notebook performs inverse identification of time-temperature shift parameters over the full non-isothermal shape-memory cycle using LSTM-based RT-RPINN.


## Import Required Libraries


In [ ]:
import sys
import math
import time
import re
import argparse
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path


# ---------------------------------------------------------------------------
# Tee: mirror stdout to log file
# ---------------------------------------------------------------------------
class _Tee:
    def __init__(self, original, file_path):
        self._orig = original
        self._file = open(file_path, 'w', buffering=1, encoding='utf-8')

    def write(self, data):
        self._orig.write(data)
        self._file.write(data)

    def flush(self):
        self._orig.flush()
        self._file.flush()

    def close(self):
        self._file.close()

    def __getattr__(self, name):
        return getattr(self._orig, name)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ---------------------------------------------------------------------------
# 1. FE Data Loader — multi-step non-isothermal cycle
# ---------------------------------------------------------------------------

# Temperature protocol derived from Abaqus step definitions


## FEDataLoader


In [ ]:
_STEP_T_FN = {
    1: lambda t: 343.0,
    2: lambda t: 343.0 - (343.0 - 298.0) * (t - 50.0) / 50.0,
    3: lambda t: 298.0,
    4: lambda t: 298.0 + (353.0 - 298.0) * (t - 105.0) / 50.0,
}
# Step time boundaries
_STEP_BOUNDS = {1: (0.0, 50.0), 2: (50.0, 100.0), 3: (100.0, 105.0), 4: (105.0, 155.0)}
# Time during Step-2 cooling when T crosses T_cross=317.4 K (WLF→Arrhenius boundary)
# t = 50 + (343-317.4)/(343-298)*50 = 78.4 s
_T_CROSS_TIME = 78.4
_STEP_PATTERNS = {
    1: '*_Step-1-High-temperature_frame*.csv',
    2: '*_Step-2-Cooling_frame*.csv',
    3: '*_Step-3-Low-temperature unloading_frame*.csv',
    4: '*_Step-4-Free recovery_frame*.csv',
}

class FEDataLoader:
    """Load and process FE simulation results from multi-step non-isothermal cycle."""

    def __init__(self, data_dir, rf_file, u_file, frame_time_file, stride=5):
        """
        Args:
            stride: Load every stride-th frame per step (default 5).
                    Frame 0 and the last frame of each step are always kept
                    to preserve step transitions.
                    stride=1 → all 1571 frames (slow, ~1571 CSVs read)
                    stride=5 → ~315 frames  (recommended for training)
                    stride=10 → ~157 frames (faster, coarser)
        """
        self.data_dir = Path(data_dir)
        self.rf_file = Path(rf_file)
        self.u_file = Path(u_file)
        self.frame_time_file = Path(frame_time_file)
        self.stride = max(1, int(stride))
        self.data = []
        self.step_frame_time = {}  # (step, local_frame) → time
        self.all_frame_times = None  # Full FE timeline from frame-time.csv

    def load_all_data(self):
        """Load (strided) CSV files from the 4-step results directory."""
        self.step_load_stats = {}

        # ------------------------------------------------------------------
        # Build (step, local_frame) → time mapping
        # frame-time.csv contains rows like "local_frame, time" for each step
        # concatenated; step boundaries detected by local_frame resetting to 1.
        # ------------------------------------------------------------------
        raw_ft = pd.read_csv(self.frame_time_file, header=None,
                              names=['LocalFrame', 'Time'])
        step = 1
        prev_frame = 0
        step_ids = []
        for _, row in raw_ft.iterrows():
            lf = int(row['LocalFrame'])
            if lf < prev_frame:
                step += 1
            step_ids.append(step)
            prev_frame = lf
        raw_ft['Step'] = step_ids
        self.step_frame_time = {
            (int(r['Step']), int(r['LocalFrame'])): float(r['Time'])
            for _, r in raw_ft.iterrows()
        }
        # Add frame-0 entries (initial conditions, not in frame-time.csv)
        step_start_t = {1: 0.0, 2: 50.0, 3: 100.0, 4: 105.0}
        for s, t0 in step_start_t.items():
            self.step_frame_time[(s, 0)] = t0
        self.all_frame_times = np.array(
            sorted(set(float(t) for t in self.step_frame_time.values())),
            dtype=float
        )

        # ------------------------------------------------------------------
        # Load per-frame CSVs for each step
        # ------------------------------------------------------------------
        def frame_from_filename(fp):
            m = re.search(r'frame(\d+)', Path(fp).stem)
            if not m:
                raise ValueError(f"Cannot parse frame index from: {fp}")
            return int(m.group(1))

        for step_num, pattern in _STEP_PATTERNS.items():
            step_files = sorted(
                list(self.data_dir.glob(pattern)),
                key=lambda f: frame_from_filename(f)
            )
            if not step_files:
                continue

            self.step_load_stats[step_num] = {
                'n_loaded_frames': 0,
                't_min': float('inf'),
                't_max': float('-inf'),
                'T_min': float('inf'),
                'T_max': float('-inf'),
                'n_out_of_bound_time': 0,
            }

            # Always keep first and last frame; apply stride to the rest
            last_idx = len(step_files) - 1
            keep = set()
            keep.add(0)
            keep.add(last_idx)
            keep.update(range(0, last_idx + 1, self.stride))
            keep_sorted = sorted(keep)

            for file_idx in keep_sorted:
                csv_file = step_files[file_idx]
                local_frame = frame_from_filename(csv_file)
                time_val = self.step_frame_time.get((step_num, local_frame), None)
                if time_val is None:
                    continue  # No time mapping → skip

                df = pd.read_csv(csv_file)
                df['Time'] = time_val
                df['Frame'] = local_frame
                df['Step'] = step_num
                T_fn = _STEP_T_FN[step_num]
                T_val = float(T_fn(time_val))
                if step_num == 2:
                    T_val = min(343.0, max(298.0, T_val))
                elif step_num == 4:
                    T_val = min(353.0, max(298.0, T_val))
                df['Temperature'] = T_val
                self.data.append(df)

                # Step-level load sanity stats
                st = self.step_load_stats[step_num]
                st['n_loaded_frames'] += 1
                st['t_min'] = min(st['t_min'], float(time_val))
                st['t_max'] = max(st['t_max'], float(time_val))
                st['T_min'] = min(st['T_min'], float(T_val))
                st['T_max'] = max(st['T_max'], float(T_val))
                t_lo, t_hi = _STEP_BOUNDS[step_num]
                if (time_val < t_lo - 1e-3) or (time_val > t_hi + 1e-3):
                    st['n_out_of_bound_time'] += 1

        if not self.data:
            raise RuntimeError(f"No data loaded from {self.data_dir}. "
                               "Check step CSV filenames match expected patterns.")

        # Sort by time to ensure chronological order
        self.data.sort(key=lambda df: df['Time'].iloc[0])

        self.full_data = pd.concat(self.data, ignore_index=True)

        # ------------------------------------------------------------------
        # Load RF and U history
        # ------------------------------------------------------------------
        rf_df = pd.read_csv(self.rf_file, header=None, names=['Time', 'RF'])
        u_df = pd.read_csv(self.u_file, header=None, names=['Time', 'U'])

        # Match RF/U times to the COMPLETE FE timeline, not the strided field subset.
        # Otherwise stride>1 creates artificial time misalignment warnings.
        all_times = self.all_frame_times

        def match_to_times(query_times, tol=1e-4):
            query_times = np.asarray(query_times, dtype=float)
            idx = np.searchsorted(all_times, query_times)
            idx = np.clip(idx, 1, len(all_times) - 1)
            left = idx - 1
            choose_right = (np.abs(all_times[idx] - query_times) <
                            np.abs(all_times[left] - query_times))
            best = np.where(choose_right, idx, left)
            matched = all_times[best]
            err = np.abs(matched - query_times)
            if np.any(err > tol):
                n_bad = int(np.sum(err > tol))
                print(f"  [Warning] {n_bad} RF/U times exceed tol={tol:.2e}. "
                      f"Max err={err.max():.3e}")
            return matched

        rf_df['MatchedTime'] = match_to_times(rf_df['Time'].values)
        u_df['MatchedTime'] = match_to_times(u_df['Time'].values)
        self.rf_data = rf_df.rename(columns={'MatchedTime': 'AlignedTime'})
        self.u_data = u_df.rename(columns={'MatchedTime': 'AlignedTime'})

        print(f"\nFE data loaded (stride={self.stride}):")
        print(f"  Total frames : {len(self.data)}")
        print(f"  Total nodes  : {len(self.full_data['NodeLabel'].unique())} unique")
        print(f"  Time range   : [{self.full_data['Time'].min():.2f}, "
              f"{self.full_data['Time'].max():.2f}] s")
        print(f"  Temp range   : [{self.full_data['Temperature'].min():.1f}, "
              f"{self.full_data['Temperature'].max():.1f}] K")
        print(f"  RF range     : [{rf_df['RF'].min():.2f}, {rf_df['RF'].max():.2f}] N")
        print("  Step summary :")
        for s in sorted(self.step_load_stats.keys()):
            st = self.step_load_stats[s]
            print(
                f"    Step{s}: frames={st['n_loaded_frames']} "
                f"t=[{st['t_min']:.2f},{st['t_max']:.2f}] "
                f"T=[{st['T_min']:.1f},{st['T_max']:.1f}] "
                f"out_of_bound_time={st['n_out_of_bound_time']}"
            )

        return self.full_data

    def get_domain_bounds(self):
        return {
            'x_min': self.full_data['X'].min(),
            'x_max': self.full_data['X'].max(),
            'y_min': self.full_data['Y'].min(),
            'y_max': self.full_data['Y'].max(),
            'z_min': self.full_data['Z'].min(),
            'z_max': self.full_data['Z'].max(),
            't_min': self.full_data['Time'].min(),
            't_max': self.full_data['Time'].max(),
            'T_min': self.full_data['Temperature'].min(),
            'T_max': self.full_data['Temperature'].max(),
        }

    def get_rf_by_time(self, time_val, tol=1e-4):
        """Return RF value closest to time_val."""
        diff = np.abs(self.rf_data['AlignedTime'].values - time_val)
        idx = int(np.argmin(diff))
        return float(self.rf_data['RF'].iloc[idx])

    def get_u_by_time(self, time_val, tol=1e-4):
        """Return loaded-end displacement closest to time_val."""
        diff = np.abs(self.u_data['AlignedTime'].values - time_val)
        idx = int(np.argmin(diff))
        return float(self.u_data['U'].iloc[idx])


# ---------------------------------------------------------------------------
# 2. Trainable Shift Parameters
# ---------------------------------------------------------------------------


## Model and Material Parameters


In [ ]:
class InverseShiftParams(nn.Module):
    """
    Trainable time-temperature shift parameters for inverse identification.

    Trainable (3 DOF, independent log-parameterisation):
      log_C1:    WLF parameter C1  (init: log(8.0),     true: log(14.8))
      log_C2:    WLF parameter C2  (init: log(25.0),    true: log(45.6))
      log_Ea_R:  Arrhenius Ea/R    (init: log(20000.0), true: log(27403.3))

    Fixed:
      All elastic constants and Prony spectrum (from EX2 / UMAT ground truth)
      T_ref   = 323 K   (WLF reference temperature)
      T_arr   = 336 K   (Arrhenius reference temperature)
      T_cross = 317.4 K (WLF/Arrhenius crossover temperature)
    """

    # ---- Ground-truth values (for logging; not used in forward pass) ----
    TRUE_C1    = 14.8
    TRUE_C2    = 45.6
    TRUE_Ea_R  = 27403.3
    TRUE_T_REF = 323.0
    TRUE_T_ARR = 336.0
    TRUE_T_CROSS = 317.4

    def __init__(self):
        super().__init__()

        # ------------------------------------------------------------------
        # Trainable shift parameters (log parameterisation → always positive)
        # Independent C1 and C2 — decoupled gradients avoid kappa-coupling drift.
        # Moderately low init: still non-trivial inverse, but avoids severe
        # early-stage drift and poor local minima in no-RF mode.
        # ------------------------------------------------------------------
        self.log_C1    = nn.Parameter(torch.tensor(math.log(8.0),     dtype=torch.float32))
        self.log_C2    = nn.Parameter(torch.tensor(math.log(25.0),    dtype=torch.float32))
        # Ea_R init: 20000 K (~73% of true 27403 K).
        # Previously 15000 caused near-zero gradient in ear_only phase (aT already saturated).
        # 20000 is still "wrong enough" to observe convergence but gives stronger gradient signal.
        self.log_Ea_R  = nn.Parameter(torch.tensor(math.log(20000.0), dtype=torch.float32))

        # ------------------------------------------------------------------
        # Fixed physical constants
        # ------------------------------------------------------------------
        self.T_ref  = 323.0   # WLF reference [K]
        self.T_arr  = 336.0   # Arrhenius reference [K]
        self.T_cross = 317.4  # crossover [K]

        # ------------------------------------------------------------------
        # Fixed Prony spectrum (5 branches; g5@10000s merged into g_inf)
        # True: g0=0.206, g1=0.093, g2=0.306, g3=0.358, g4=0.034, g_inf=0.002+0.001
        # ------------------------------------------------------------------
        self.N_prony = 5
        self.register_buffer("rho",
            torch.tensor([0.1, 1.0, 10.0, 100.0, 1000.0], dtype=torch.float32))
        self.register_buffer("g_weights",
            torch.tensor([0.206, 0.093, 0.306, 0.358, 0.034, 0.003], dtype=torch.float32))
        # g_weights[5] = g_inf = 0.002 + 0.001 = 0.003 (g5 merged)

        # ------------------------------------------------------------------
        # Fixed elastic constants [MPa] (from True_Material_Parameters.md)
        # ------------------------------------------------------------------
        self.register_buffer("C11_0", torch.tensor(11250.16, dtype=torch.float32))
        self.register_buffer("C12_0", torch.tensor(891.26,   dtype=torch.float32))
        self.register_buffer("C22_0", torch.tensor(1425.85,  dtype=torch.float32))
        self.register_buffer("C23_0", torch.tensor(915.56,   dtype=torch.float32))
        self.register_buffer("C66_0", torch.tensor(267.69,   dtype=torch.float32))

        # Long-term stiffness C∞
        # C11∞ is hardcoded from UMAT (fiber-dominated, different from matrix branches).
        # For matrix components (C12, C22, C23, C66) the true g_inf = 0.002.
        # g_weights[-1] = 0.003 is g5(τ=10000s)+g_inf merged — using it here
        # overestimates C∞ by 50% (2.67 vs 1.77 MPa for C12).
        g_inf_mat = 0.002   # true equilibrium fraction for matrix stiffness components
        self.register_buffer("C11_inf", 10319.10 * torch.ones(1, dtype=torch.float32))
        self.register_buffer("C12_inf", (g_inf_mat * self.C12_0).unsqueeze(0))
        self.register_buffer("C22_inf", (g_inf_mat * self.C22_0).unsqueeze(0))
        self.register_buffer("C23_inf", (g_inf_mat * self.C23_0).unsqueeze(0))
        self.register_buffer("C66_inf", (g_inf_mat * self.C66_0).unsqueeze(0))

        # 45° rotation buffer
        c = math.cos(math.radians(45.0))
        s = math.sin(math.radians(45.0))
        self.register_buffer("Q", torch.tensor([
            [ c, -s, 0.],
            [ s,  c, 0.],
            [ 0., 0., 1.]
        ], dtype=torch.float32))

    # ------------------------------------------------------------------ properties

    @property
    def C1(self):
        return torch.exp(self.log_C1)

    @property
    def C2(self):
        return torch.exp(self.log_C2)

    @property
    def Ea_R(self):
        return torch.exp(self.log_Ea_R)

    def shift_factor(self, T_tensor):
        """
        Hybrid WLF / Arrhenius shift factor a_T(T).

        Args:
            T_tensor: Temperature tensor, shape (..., 1) [K]

        Returns:
            a_T tensor of same shape (always > 0)
        """
        T = T_tensor
        C1 = self.C1
        C2 = self.C2
        Ea_R = self.Ea_R
        T_ref  = self.T_ref
        T_arr  = self.T_arr
        T_cross = self.T_cross

        # WLF (T > T_cross): log10(a_T) = -C1*(T-Tref)/(C2+(T-Tref))
        # Use explicit clamping + exp(log(10)*x) for better gradient stability.
        delta_T = T - T_ref
        denom = C2 + delta_T
        denom_safe = torch.clamp(denom, min=5.0, max=300.0)
        log10_aT_wlf = -C1 * delta_T / denom_safe
        log10_aT_wlf = torch.clamp(log10_aT_wlf, min=-12.0, max=12.0)
        aT_wlf = torch.exp(math.log(10.0) * log10_aT_wlf)

        # Arrhenius (T ≤ T_cross): a_T = exp(Ea_R*(1/T - 1/T_arr))
        T_safe = torch.clamp(T, min=200.0)  # avoid 1/T → inf
        arr_exponent = Ea_R * (1.0 / T_safe - 1.0 / T_arr)
        arr_exponent = torch.clamp(arr_exponent, min=-80.0, max=80.0)
        aT_arr = torch.exp(arr_exponent)

        # Blend smoothly around T_cross to avoid gradient spikes near the switch.
        blend = torch.sigmoid((T - T_cross) / 1.5)
        aT = blend * aT_wlf + (1.0 - blend) * aT_arr

        # Safety clamp: prevent extreme values that cause NaN in q recursion
        aT = torch.clamp(aT, min=1e-10, max=1e15)
        return aT

    def get_stiffness_matrix_inf(self):
        """Construct C^∞ stiffness matrix (6×6 Voigt)."""
        dev = self.C11_inf.device
        C11 = self.C11_inf[0]
        C12 = self.C12_inf[0]
        C22 = self.C22_inf[0]
        C23 = self.C23_inf[0]
        C44 = (C22 - C23) / 2.0
        C66 = self.C66_inf[0]

        Cm = torch.zeros(6, 6, device=dev, dtype=torch.float32)
        Cm[0,0] = C11; Cm[1,1] = C22; Cm[2,2] = C22
        Cm[0,1] = C12; Cm[1,0] = C12
        Cm[0,2] = C12; Cm[2,0] = C12
        Cm[1,2] = C23; Cm[2,1] = C23
        Cm[3,3] = C66; Cm[4,4] = C66; Cm[5,5] = C44
        return Cm

    def get_stiffness_matrix_branch(self, k):
        """Construct Prony branch k stiffness matrix C^(k) (6×6 Voigt)."""
        dev = self.C11_0.device
        gk = self.g_weights[k]
        # C11 branch uses its own spectrum (fiber-dominated, ~8% relaxation)
        # Approximate: use full C(0) × g_k for simplicity
        C11k = gk * (self.C11_0 - self.C11_inf[0])  # branch contribution from C11
        C12k = gk * self.C12_0
        C22k = gk * self.C22_0
        C23k = gk * self.C23_0
        C66k = gk * self.C66_0
        C44k = (C22k - C23k) / 2.0

        Ck = torch.zeros(6, 6, device=dev, dtype=torch.float32)
        Ck[0,0] = C11k; Ck[1,1] = C22k; Ck[2,2] = C22k
        Ck[0,1] = C12k; Ck[1,0] = C12k
        Ck[0,2] = C12k; Ck[2,0] = C12k
        Ck[1,2] = C23k; Ck[2,1] = C23k
        Ck[3,3] = C66k; Ck[4,4] = C66k; Ck[5,5] = C44k
        return Ck

    def compute_parameter_penalty(self):
        """Soft constraints to keep shift parameters in physical range."""
        dev = self.log_C1.device
        penalty = torch.zeros((), device=dev, dtype=torch.float32)
        C1 = self.C1
        C2 = self.C2
        Ea_R = self.Ea_R
        # C1 range [4, 30]
        penalty = penalty + 0.20 * torch.relu(4.0 - C1) ** 2
        penalty = penalty + 0.20 * torch.relu(C1 - 30.0) ** 2
        # C2 range [10, 100]
        penalty = penalty + 0.08 * torch.relu(10.0 - C2) ** 2
        penalty = penalty + 0.08 * torch.relu(C2 - 100.0) ** 2
        # Ea/R range [3000, 50000]
        penalty = penalty + 1e-5 * torch.relu(3000.0 - Ea_R) ** 2
        penalty = penalty + 1e-5 * torch.relu(Ea_R - 50000.0) ** 2
        # Physical continuity at crossover: aT_WLF(T_cross) = aT_Arrhenius(T_cross).
        # This equation links C1, C2, and Ea_R:
        #   -C1*(T_cross-Tref)/(C2+(T_cross-Tref)) = (Ea_R/ln10)*(1/T_cross-1/T_arr)
        # Weight raised from 1e-3 → 0.15 so the penalty (~7.5 at wrong init) is
        # comparable to other loss terms (~0.05-0.5), providing a strong identifiability
        # signal for Ea_R during ear_only phase instead of zero gradient.
        delta_cross = self.T_cross - self.T_ref
        denom_cross = torch.clamp(C2 + delta_cross, min=5.0, max=300.0)
        log10_a_wlf_cross = -C1 * delta_cross / denom_cross
        ln_a_wlf_cross = math.log(10.0) * log10_a_wlf_cross
        ln_a_arr_cross = Ea_R * (1.0 / self.T_cross - 1.0 / self.T_arr)
        penalty = penalty + 0.15 * (ln_a_wlf_cross - ln_a_arr_cross) ** 2
        return penalty

    def load_state_dict_compat(self, state_dict):
        """Load checkpoints in both kappa-based (legacy) and direct C1 (current) formats."""
        sd = dict(state_dict)
        if ('log_C1' not in sd) and ('log_kappa' in sd) and ('log_C2' in sd):
            # Legacy format: C1 = kappa * C2 → log_C1 = log_kappa + log_C2
            sd['log_C1'] = sd['log_kappa'] + sd['log_C2']
            sd.pop('log_kappa', None)
        self.load_state_dict(sd, strict=False)


# ---------------------------------------------------------------------------
# 3. PINN — 6D input (x,y,z, fiber coords, t, T)
# ---------------------------------------------------------------------------

# ---------------------------------------------------------------------------
# 4. Inverse PINN Solver
# ---------------------------------------------------------------------------



class LSTMInverseEX4(nn.Module):
    """
    LSTM-PINN for EX4 inverse shift parameter identification.

    Input sequence per time step:
      Spatial features from (x_hat, y_hat, z_hat, x_local, y_local, T_hat) → MLP → features
    LSTM processes the temporal sequence, decoder maps to (u1, u2, u3).

    Args:
        spatial_hidden: Hidden sizes for the spatial MLP encoder
        lstm_hidden:    LSTM hidden state size
        lstm_layers:    Number of LSTM layers
        dropout:        Dropout probability
    """

    def __init__(self, spatial_hidden=(64, 64), lstm_hidden=64,
                 lstm_layers=2, dropout=0.0):
        super().__init__()

        self.lstm_hidden = lstm_hidden
        self.lstm_layers = lstm_layers

        c = math.cos(math.radians(45.0))
        s = math.sin(math.radians(45.0))
        self.register_buffer("cos_theta", torch.tensor(c, dtype=torch.float32))
        self.register_buffer("sin_theta", torch.tensor(s, dtype=torch.float32))

        # Spatial encoder: 6D (x,y,z,x_local,y_local,T) → features
        dims = [6, *spatial_hidden]
        mlp = []
        for i in range(len(dims) - 1):
            mlp.append(nn.Linear(dims[i], dims[i + 1]))
            mlp.append(nn.Tanh())
            if dropout > 0:
                mlp.append(nn.Dropout(dropout))
        self.spatial_encoder = nn.Sequential(*mlp)

        # LSTM
        self.lstm = nn.LSTM(
            input_size=dims[-1],
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            dropout=dropout if lstm_layers > 1 else 0.0,
        )

        # Decoder: hidden → displacement (u1,u2,u3)
        self.decoder = nn.Sequential(
            nn.Linear(lstm_hidden, lstm_hidden),
            nn.Tanh(),
            nn.Linear(lstm_hidden, 3),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def _encode_spatial(self, x, y, z, T):
        """
        Encode (x,y,z, fiber-local, T) to spatial features.

        Args:
            x, y, z, T: shape (batch, seq, 1) or (batch, 1)
        Returns:
            features: same leading dims → (batch, seq, feature_dim)
        """
        x_local = x * self.cos_theta + y * self.sin_theta
        y_local = -x * self.sin_theta + y * self.cos_theta
        h = torch.cat([x, y, z, x_local, y_local, T], dim=-1)
        return self.spatial_encoder(h)

    def forward_sequence(self, x, y, z, T, h0=None, c0=None):
        """
        Process a temporal sequence.

        Args:
            x, y, z, T: Normalised coords, shape (batch, seq, 1)
            h0, c0:     Optional initial LSTM hidden state

        Returns:
            u_seq:    Displacements (batch, seq, 3)
            (h, c):  Final hidden state
        """
        spatial_feats = self._encode_spatial(x, y, z, T)  # (B, S, F)
        lstm_out, (h, c) = self.lstm(
            spatial_feats,
            (h0, c0) if h0 is not None else None
        )  # (B, S, lstm_hidden)
        u_seq = self.decoder(lstm_out)  # (B, S, 3)
        return u_seq, (h, c)

    def forward_step(self, x, y, z, T, h, c):
        """
        Process ONE time step with given hidden state.

        Used in the training loop to make per-step strain computation O(1) instead
        of O(N_t) per step (which was the source of the 24-hour hang: full-sequence
        autograd.grad through N_t LSTM steps, called N_t times = O(N_t²) total).

        Args:
            x, y, z, T: (batch, 1) — spatial coords and temperature for this step
            h, c:       (lstm_layers, batch, lstm_hidden) — LSTM hidden state

        Returns:
            u:        (batch, 3) displacement at this step
            (h, c):  Updated hidden state
        """
        # unsqueeze seq dimension → (batch, 1, feat)
        feats = self._encode_spatial(
            x.unsqueeze(1), y.unsqueeze(1), z.unsqueeze(1), T.unsqueeze(1)
        )
        out, (h_new, c_new) = self.lstm(feats, (h, c))   # out: (batch, 1, H)
        u = self.decoder(out[:, 0, :])                    # (batch, 3)
        return u, (h_new, c_new)


# ---------------------------------------------------------------------------
# LSTM Inverse Solver — inherits physics from InversePINNSolver
# ---------------------------------------------------------------------------


## Solver and Loss Functions


In [ ]:
class InversePINNSolver:
    """
    Solver for inverse WLF/Arrhenius shift parameter identification.

    Physics:
      σ = C^∞ : ε + Σ_k C^(k) : (ε - q^(k))
      q^(k)_n = exp(-Δξ_n/ρ_k)*q^(k)_{n-1} + (1 - exp(-Δξ_n/ρ_k))*ε_n
      Δξ_n = Δt_n / a_T(T_n)     ← key difference vs EX2

    All strain in MATERIAL coordinates (45° rotated).
    """

    def __init__(self, model, mat, fe_loader, bounds,
                 lambda_data=10.0, lambda_strain=10.0, lambda_stress=5.0,
                 lambda_pde=5.0,
                 lambda_bc_left=1.0, lambda_bc_right=10.0,
                 lambda_traction=5.0, lambda_rf=5.0, lambda_obs=20.0,
                 lambda_obs_rate=0.8):
        self.model = model.to(device)
        self.mat = mat.to(device)
        self.fe_loader = fe_loader
        self.bounds = bounds

        self.lambda_data = lambda_data
        self.lambda_strain = lambda_strain
        self.lambda_stress = lambda_stress
        self.lambda_pde = lambda_pde
        self.lambda_bc_left = lambda_bc_left
        self.lambda_bc_right = lambda_bc_right
        self.lambda_traction = lambda_traction
        self.lambda_rf = lambda_rf
        self.lambda_obs = lambda_obs
        self.lambda_obs_rate = lambda_obs_rate

        # Derived domain lengths
        self.x_min = bounds['x_min']; self.Lx = bounds['x_max'] - bounds['x_min']
        self.y_min = bounds['y_min']; self.Ly = bounds['y_max'] - bounds['y_min']
        self.z_min = bounds['z_min']; self.Lz = bounds['z_max'] - bounds['z_min']
        self.t_min = bounds['t_min']; self.T_max = bounds['t_max'] - bounds['t_min']
        self.Temp_min = bounds['T_min']
        self.Temp_range = max(bounds['T_max'] - bounds['T_min'], 1.0)

        self.loss_history = []
        self.loss_components_history = []
        self.param_history = []
        self.u_ref_sq = 1.0
        self.rf_rel_floor = torch.tensor(100.0, device=device)
        self.rf_node_ids = None
        self.rf_node_col = None

    # ------------------------------------------------------------------
    # Normalisation helpers
    # ------------------------------------------------------------------
    def norm_coords(self, x, y, z, t, T):
        x_hat = (x - self.x_min) / self.Lx
        y_hat = (y - self.y_min) / self.Ly
        z_hat = (z - self.z_min) / self.Lz
        t_hat = (t - self.t_min) / self.T_max
        T_hat = (T - self.Temp_min) / self.Temp_range
        return x_hat, y_hat, z_hat, t_hat, T_hat

    # ------------------------------------------------------------------
    # Rotation: global ↔ material coordinates (45° about Z)
    # ------------------------------------------------------------------
    def to_material_coords(self, strain_global):
        """Rotate engineering strain (N,6) from global to 45° material frame."""
        Q = self.mat.Q
        N = strain_global.shape[0]
        e11, e22, e33 = strain_global[:, 0], strain_global[:, 1], strain_global[:, 2]
        g12, g13, g23 = strain_global[:, 3], strain_global[:, 4], strain_global[:, 5]
        e12 = g12 / 2.0; e13 = g13 / 2.0; e23 = g23 / 2.0

        # Full strain tensor
        e = torch.stack([
            torch.stack([e11, e12, e13], dim=1),
            torch.stack([e12, e22, e23], dim=1),
            torch.stack([e13, e23, e33], dim=1),
        ], dim=2)  # (N,3,3)

        # Rotate: ε_mat = Q^T ε Q
        QT = Q.t()
        e_mat = torch.einsum('ij,njk,kl->nil', QT, e, Q)

        e11m = e_mat[:, 0, 0]; e22m = e_mat[:, 1, 1]; e33m = e_mat[:, 2, 2]
        e12m = e_mat[:, 0, 1]; e13m = e_mat[:, 0, 2]; e23m = e_mat[:, 1, 2]

        return torch.stack([e11m, e22m, e33m,
                            2.0*e12m, 2.0*e13m, 2.0*e23m], dim=1)

    # ------------------------------------------------------------------
    # Strain via autodiff
    # ------------------------------------------------------------------
    def compute_strain(self, u, x_leaf, y_leaf, z_leaf):
        """Engineering strain (N,6) from displacement (N,3) via autodiff."""
        u1, u2, u3 = u[:, 0:1], u[:, 1:2], u[:, 2:3]

        def _grad(ui, xi):
            return torch.autograd.grad(ui, xi, torch.ones_like(ui),
                                       create_graph=True, retain_graph=True)[0]

        e11 = _grad(u1, x_leaf)
        e22 = _grad(u2, y_leaf)
        e33 = _grad(u3, z_leaf)
        g12 = _grad(u1, y_leaf) + _grad(u2, x_leaf)
        g13 = _grad(u1, z_leaf) + _grad(u3, x_leaf)
        g23 = _grad(u2, z_leaf) + _grad(u3, y_leaf)

        return torch.cat([e11, e22, e33, g12, g13, g23], dim=1)

    # ------------------------------------------------------------------
    # Q recursion with reduced time (NON-ISOTHERMAL)
    # ------------------------------------------------------------------
    @staticmethod
    def compute_q_recursive(strain_mat_seq, dt_seq, T_seq, mat):
        """
        Internal-variable recursion with WLF reduced time.

        Args:
            strain_mat_seq: (N_t, N_pts, 6) material-frame strain
            dt_seq:         (N_t-1,) real-time increments [s]
            T_seq:          (N_t, N_pts) temperature at each time step [K]
            mat:            InverseShiftParams (provides shift_factor())

        Returns:
            q_seq: (N_t, N_pts, 6, N_prony)
        """
        N_t, N_pts, _ = strain_mat_seq.shape
        N_prony = mat.N_prony
        dev = strain_mat_seq.device

        q_list = [
            torch.zeros(N_pts, 6, N_prony, device=dev, dtype=strain_mat_seq.dtype)
        ]
        rho = mat.rho.view(1, 1, N_prony)  # (1,1,N_prony)
        truncate_every = 50

        for n in range(1, N_t):
            dt = dt_seq[n - 1]  # scalar
            strain_n = strain_mat_seq[n]  # (N_pts, 6)
            T_n = T_seq[n].reshape(N_pts, 1)  # (N_pts, 1)

            # Shift factor and reduced time increment
            aT_n = mat.shift_factor(T_n)           # (N_pts, 1)
            aT_n = torch.nan_to_num(aT_n, nan=1.0, posinf=1e12, neginf=1e-8)
            aT_n = torch.clamp(aT_n, min=1e-8, max=1e12)

            delta_xi = (dt / aT_n).unsqueeze(-1)   # (N_pts, 1, 1)
            delta_xi = torch.nan_to_num(delta_xi, nan=0.0, posinf=1e6, neginf=0.0)
            dxi_over_rho = delta_xi / rho          # (N_pts, 1, N_prony) via broadcast
            dxi_over_rho = torch.clamp(dxi_over_rho, min=0.0, max=80.0)

            exp_factor   = torch.exp(-dxi_over_rho)  # (N_pts, 1, N_prony)
            relax_factor = 1.0 - exp_factor

            q_prev = q_list[-1]  # (N_pts, 6, N_prony)
            if n % truncate_every == 0:
                q_prev = q_prev.detach()

            # Broadcasting: strain_n (N_pts,6) → (N_pts,6,1) × relax_factor (N_pts,1,N_prony)
            strain_safe = torch.nan_to_num(strain_n, nan=0.0, posinf=1e6, neginf=-1e6)
            q_n = exp_factor * q_prev + strain_safe.unsqueeze(-1) * relax_factor
            q_n = torch.nan_to_num(q_n, nan=0.0, posinf=1e6, neginf=-1e6)
            q_list.append(q_n)

        return torch.stack(q_list, dim=0)

    # ------------------------------------------------------------------
    # Stress from strain and q (material coordinates → rotate to global)
    # ------------------------------------------------------------------
    def compute_stress(self, strain_mat, q_n):
        """
        Stress in GLOBAL coordinates via Prony branch sum.

        σ_mat = C^∞:ε_mat + Σ_k C^(k):(ε_mat - q^(k))
        σ_global = Q σ_mat Q^T

        Args:
            strain_mat: (N_pts, 6) engineering strain in material frame
            q_n:        (N_pts, 6, N_prony) internal variables in material frame

        Returns:
            stress_global: (N_pts, 6) Voigt stress in global frame
        """
        C_inf = self.mat.get_stiffness_matrix_inf()  # (6,6)
        sigma_mat = strain_mat @ C_inf.t()  # (N_pts, 6)

        for k in range(self.mat.N_prony):
            Ck = self.mat.get_stiffness_matrix_branch(k)  # (6,6)
            q_k = q_n[:, :, k]  # (N_pts, 6)
            sigma_mat = sigma_mat + (strain_mat - q_k) @ Ck.t()

        # Rotate from material to global
        Q = self.mat.Q
        N = sigma_mat.shape[0]

        # Unpack Voigt (engineering convention)
        s11, s22, s33 = sigma_mat[:, 0], sigma_mat[:, 1], sigma_mat[:, 2]
        s12, s13, s23 = sigma_mat[:, 3], sigma_mat[:, 4], sigma_mat[:, 5]

        sig_t = torch.stack([
            torch.stack([s11, s12, s13], dim=1),
            torch.stack([s12, s22, s23], dim=1),
            torch.stack([s13, s23, s33], dim=1),
        ], dim=2)  # (N,3,3)

        sig_g = torch.einsum('ij,njk,kl->nil', Q, sig_t, Q.t())

        sg11 = sig_g[:, 0, 0]; sg22 = sig_g[:, 1, 1]; sg33 = sig_g[:, 2, 2]
        sg12 = sig_g[:, 0, 1]; sg13 = sig_g[:, 0, 2]; sg23 = sig_g[:, 1, 2]

        return torch.stack([sg11, sg22, sg33, sg12, sg13, sg23], dim=1)

    def compute_reaction_force(self, stress_rf):
        """RF = mean(σ11) * Area (right face, 6×2 mm²)."""
        area = self.bounds['y_max'] * self.bounds['z_max']
        return stress_rf[:, 0].mean() * area

    # ------------------------------------------------------------------
    # Initialise fixed RF node set (stable across epochs)
    # ------------------------------------------------------------------
    def initialize_fixed_rf_points(self, n_rf_points=512):
        full_data = self.fe_loader.full_data
        x_max = self.bounds['x_max']
        right_nodes = full_data[np.abs(full_data['X'].values - x_max) < 1e-4]

        for col in ['NodeLabel', 'Node', 'NID']:
            if col in right_nodes.columns:
                self.rf_node_col = col
                node_ids = right_nodes[col].unique()
                np.random.shuffle(node_ids)
                self.rf_node_ids = node_ids[:n_rf_points]
                if self.lambda_rf > 0.0:
                    print(f"  RF fixed nodes: {len(self.rf_node_ids)} ({col})")
                else:
                    print(f"  Right-face fixed nodes: {len(self.rf_node_ids)} ({col}) [for obs/bc]")
                return
        self.rf_node_ids = None
        self.rf_node_col = None

    # ------------------------------------------------------------------
    # Temporal sequence sampler
    # ------------------------------------------------------------------
    def sample_temporal_sequence(self, full_data, batch_size, seq_length=60,
                                  required_node_ids=None, node_id_col=None):
        """
        Sample time subsequence covering all 4 cycle phases.

        Dense sampling at:
          (0,  10):   early programming ramp (τ=0.1s zone)
          (45, 55):   end of programming + start of cooling
          (50, 100):  cooling — shift factor changes rapidly (KEY for C1/C2)
          (95, 110):  end of cooling + unloading spring-back
          (105,155):  recovery heating (KEY for Ea_R + final shape recovery)
        """
        if node_id_col is None:
            for col in ['NodeLabel', 'Node', 'NID']:
                if col in full_data.columns:
                    node_id_col = col
                    break
        use_node_ids = (node_id_col is not None) and (node_id_col in full_data.columns)

        unique_times = np.sort(full_data['Time'].unique())
        N_times = len(unique_times)
        seq_length = min(seq_length, N_times)

        if seq_length >= N_times:
            time_indices = np.arange(N_times)
        else:
            # Dense zones for non-isothermal cycle (weighted).
            # Cooling [50,100] carries the strongest C1/C2 signal, so assign
            # a larger fraction of temporal budget there.
            zone_specs = [
                ((0, 10), 0.8),      # ramp onset
                ((45, 55), 1.0),     # programming-end / cooling-start
                ((50, 100), 2.4),    # constrained cooling (C1/C2 key zone)
                ((95, 110), 1.0),    # unloading transition
                ((105, 155), 1.8),   # recovery (Ea/R key zone)
            ]
            fast_budget = max(10, int(round(0.70 * seq_length)))
            w_sum = sum(w for _, w in zone_specs)
            n_uniform = max(4, seq_length - fast_budget)

            selected = {0, N_times - 1}
            for (t_lo, t_hi), w in zone_specs:
                n_zone = max(2, int(round(fast_budget * w / w_sum)))
                zone_idx = np.where((unique_times >= t_lo) & (unique_times <= t_hi))[0]
                if len(zone_idx) > 0:
                    chosen = zone_idx[
                        np.round(np.linspace(0, len(zone_idx) - 1,
                                             n_zone)).astype(int)
                    ]
                    selected.update(chosen.tolist())

            uniform_all = np.round(np.linspace(0, N_times - 1, n_uniform)).astype(int)
            selected.update(uniform_all.tolist())

            time_indices = np.sort(np.array(list(selected), dtype=int))
            if len(time_indices) > seq_length:
                keep = np.round(np.linspace(0, len(time_indices) - 1,
                                            seq_length)).astype(int)
                time_indices = time_indices[keep]

        sampled_times = unique_times[time_indices]
        assert np.all(np.diff(sampled_times) >= 0), "sampled_times not monotonic"

        def _deduplicate_time_slice(df):
            """
            Abaqus step boundaries reuse absolute times (50/100/105 s), so
            a pure Time filter can return two frames with duplicate NodeLabel.
            Keep the later (Step, Frame) state for each node to obtain one
            stable nodal snapshot per absolute time.
            """
            if len(df) == 0:
                return df
            if node_id_col is None or node_id_col not in df.columns:
                return df
            sort_cols = [c for c in ['Step', 'Frame'] if c in df.columns]
            if sort_cols:
                df = df.sort_values(sort_cols)
            return df.drop_duplicates(subset=node_id_col, keep='last')

        # ------ Spatial region identification from first frame ------
        first_slice = full_data[
            np.isclose(full_data['Time'].values, sampled_times[0], atol=1e-5)
        ]
        first_slice = _deduplicate_time_slice(first_slice)
        ref_xyz_by_node = None
        if use_node_ids and node_id_col in first_slice.columns:
            ref_xyz_by_node = first_slice.set_index(node_id_col)[['X', 'Y', 'Z']]
        x_coords = first_slice['X'].values
        y_coords = first_slice['Y'].values
        z_coords = first_slice['Z'].values
        x_min = self.bounds['x_min']; x_max = self.bounds['x_max']
        y_min = self.bounds['y_min']; y_max = self.bounds['y_max']
        z_min = self.bounds['z_min']; z_max = self.bounds['z_max']
        tol = 1e-4

        is_left_face  = np.abs(x_coords - x_min) < tol
        is_right_face = np.abs(x_coords - x_max) < tol
        is_y_face = (np.abs(y_coords - y_min) < tol) | (np.abs(y_coords - y_max) < tol)
        is_z_face = (np.abs(z_coords - z_min) < tol) | (np.abs(z_coords - z_max) < tol)
        is_yz_face = is_y_face | is_z_face
        is_any_x = is_left_face | is_right_face
        is_interior = ~is_any_x & ~is_yz_face

        left_idx     = np.where(is_left_face)[0]
        right_idx    = np.where(is_right_face)[0]
        yz_idx       = np.where(is_yz_face & ~is_any_x)[0]
        interior_idx = np.where(is_interior)[0]

        n_left     = max(5,  batch_size // 12)
        n_right    = max(10, batch_size // 7)
        n_yz_faces = max(15, batch_size // 7)
        n_interior = batch_size - n_left - n_right - n_yz_faces

        n_left     = min(n_left,     len(left_idx))
        n_right    = min(n_right,    len(right_idx))
        n_yz_faces = min(n_yz_faces, len(yz_idx))
        n_interior = min(n_interior, len(interior_idx))

        # Fixed RF nodes (if provided)
        rf_fixed_active = False
        if required_node_ids is not None and use_node_ids and len(required_node_ids) > 0:
            first_right = first_slice.iloc[right_idx]
            mask_req = first_right[node_id_col].isin(required_node_ids)
            req_found = first_right[mask_req][node_id_col].values
            if len(req_found) > 0:
                rf_fixed_active = True
                if len(req_found) >= n_right:
                    # Align with EX2 stabilization: when fixed right-face IDs are
                    # available, use all of them for RF / right-BC supervision.
                    right_node_ids = req_found
                    n_right = len(right_node_ids)
                else:
                    extra = np.random.choice(
                        first_right[~mask_req][node_id_col].values,
                        max(0, n_right - len(req_found)), replace=False
                    )
                    right_node_ids = np.concatenate([req_found, extra])[:n_right]
            else:
                rr = np.random.choice(right_idx, n_right, replace=False) if n_right > 0 else []
                right_node_ids = first_slice.iloc[rr][node_id_col].values
        else:
            rr = np.random.choice(right_idx, n_right, replace=False) if n_right > 0 else []
            right_node_ids = first_slice.iloc[rr][node_id_col].values if use_node_ids else None

        # Sample other regions
        sl = np.random.choice(left_idx, n_left, replace=False) if n_left > 0 else []
        sy = np.random.choice(yz_idx, n_yz_faces, replace=False) if n_yz_faces > 0 else []
        si = np.random.choice(interior_idx, n_interior, replace=False) if n_interior > 0 else []

        left_node_ids = first_slice.iloc[sl][node_id_col].values if (use_node_ids and len(sl)) else None
        yz_node_ids   = first_slice.iloc[sy][node_id_col].values if (use_node_ids and len(sy)) else None
        interior_node_ids = first_slice.iloc[si][node_id_col].values if (use_node_ids and len(si)) else None

        # Build combined node ID order
        all_node_ids = np.concatenate([
            right_node_ids if right_node_ids is not None else [],
            left_node_ids if left_node_ids is not None else [],
            yz_node_ids if yz_node_ids is not None else [],
            interior_node_ids if interior_node_ids is not None else [],
        ])

        rf_end   = len(right_node_ids) if right_node_ids is not None else 0
        left_end = rf_end + (len(left_node_ids) if left_node_ids is not None else 0)
        yz_end   = left_end + (len(yz_node_ids) if yz_node_ids is not None else 0)

        # Collect data per time step
        coords_seq      = []
        u_data_seq      = []
        strain_data_seq = []
        stress_data_seq = []
        T_seq_list      = []
        frames          = []

        for t_val in sampled_times:
            time_slice = full_data[np.isclose(full_data['Time'].values, t_val, atol=1e-5)]
            time_slice = _deduplicate_time_slice(time_slice)
            if len(time_slice) == 0:
                continue

            # Node-ID-aligned reindex
            if use_node_ids and node_id_col in time_slice.columns:
                ts_indexed = time_slice.set_index(node_id_col)
                ts_aligned = ts_indexed.reindex(all_node_ids)
                if ref_xyz_by_node is not None:
                    ref_xyz = ref_xyz_by_node.reindex(all_node_ids)
                    for col in ['X', 'Y', 'Z']:
                        ts_aligned[col] = ts_aligned[col].fillna(ref_xyz[col])
            else:
                ts_aligned = time_slice.iloc[np.arange(min(len(all_node_ids), len(time_slice)))]

            xyz = torch.tensor(
                ts_aligned[['X', 'Y', 'Z']].fillna(0.0).values, dtype=torch.float32, device=device
            )
            u_vals = torch.tensor(
                ts_aligned[['U1', 'U2', 'U3']].fillna(0.0).values,
                dtype=torch.float32, device=device
            )
            T_val = float(time_slice['Temperature'].iloc[0])

            le_cols = ['LE11', 'LE22', 'LE33', 'LE12', 'LE13', 'LE23']
            if all(c in ts_aligned.columns for c in le_cols):
                strain_fe = torch.tensor(
                    ts_aligned[le_cols].fillna(0.0).values,
                    dtype=torch.float32, device=device
                )
                # Convert LE (tensor shear) to engineering shear (×2 for off-diagonal)
                strain_fe_eng = strain_fe.clone()
                strain_fe_eng[:, 3:] = strain_fe[:, 3:] * 2.0
                strain_data_seq.append(strain_fe_eng)
            else:
                strain_data_seq.append(None)

            # FE stress (global Cauchy) — direct shift-parameter gradient path
            s_cols = ['S11', 'S22', 'S33', 'S12', 'S13', 'S23']
            if all(c in ts_aligned.columns for c in s_cols):
                stress_fe = torch.tensor(
                    ts_aligned[s_cols].fillna(0.0).values,
                    dtype=torch.float32, device=device
                )
                stress_data_seq.append(stress_fe)
            else:
                stress_data_seq.append(None)

            coords_seq.append(xyz)
            u_data_seq.append(u_vals)
            T_seq_list.append(T_val)

            frame_val = int(time_slice['Frame'].iloc[0]) if 'Frame' in time_slice.columns else 0
            frames.append(frame_val)

        N_t = len(coords_seq)
        N_pts = len(all_node_ids)

        # dt sequence
        actual_times = [float(full_data[
            np.isclose(full_data['Time'].values, t_val, atol=1e-5)
        ]['Time'].iloc[0]) for t_val in sampled_times[:N_t]]

        dt_list = [actual_times[n] - actual_times[n-1] for n in range(1, N_t)]
        dt_seq = torch.tensor(dt_list, dtype=torch.float32, device=device)

        right_indices  = list(range(0, rf_end))
        left_indices   = list(range(rf_end, left_end))
        yz_face_indices = list(range(left_end, yz_end))
        interior_start  = yz_end

        return {
            'coords_seq':      coords_seq,
            'u_data_seq':      u_data_seq,
            'strain_data_seq': strain_data_seq,
            'stress_data_seq': stress_data_seq,
            'times':           actual_times[:N_t],
            'T_seq':           T_seq_list[:N_t],
            'dt_seq':          dt_seq,
            'frames':          frames[:N_t],
            'N_pts':           N_pts,
            'N_t':             N_t,
            'right_indices':   right_indices,
            'left_indices':    left_indices,
            'yz_face_indices': yz_face_indices,
            'interior_start':  interior_start,
            'n_right':         rf_end,
            'n_left':          left_end - rf_end,
            'n_yz_faces':      yz_end - left_end,
            'n_interior':      N_pts - yz_end,
        }

    # ------------------------------------------------------------------
    # Main training loop
    # ------------------------------------------------------------------
    def train(self, epochs=10000, batch_size=512,
              lr_network=2e-3, lr_params=1e-2,
              log_interval=100, seq_length=60,
              adaptive_loss_balance=True, adaptive_lr=True):

        start_time = time.time()
        full_data = self.fe_loader.full_data

        # Displacement reference scale
        x_max = self.bounds['x_max']
        right_data = full_data[np.abs(full_data['X'].values - x_max) < 1e-4]
        u_mag = np.linalg.norm(right_data[['U1', 'U2', 'U3']].values, axis=1)
        u_ref = float(np.max(u_mag))
        self.u_ref_sq = (u_ref + 1e-10) ** 2

        rf_values = torch.tensor(self.fe_loader.rf_data['RF'].values,
                                  dtype=torch.float32, device=device)
        # Use a physically meaningful RF relative floor.
        # A fixed 100 N floor makes EX4 RF loss almost inactive (RF max ~17 N).
        rf_max_abs = float(torch.max(torch.abs(rf_values)).item()) if rf_values.numel() > 0 else 0.0
        rf_floor_val = max(2.0, 0.15 * rf_max_abs)
        self.rf_rel_floor = torch.tensor(rf_floor_val, dtype=torch.float32, device=device)
        print(f"\nU scale (max|u|): {u_ref:.4f} mm")
        if self.lambda_rf > 0.0:
            print(f"RF range: [{rf_values.min().item():.2f}, {rf_values.max().item():.2f}] N")
            print(f"RF relative floor: {self.rf_rel_floor.item():.2f} N")
        else:
            print("RF supervision: disabled (lambda_rf=0)")

        # Align with EX2: use all available right-face nodes for RF supervision.
        x_max = self.bounds['x_max']
        right_nodes = full_data[np.abs(full_data['X'].values - x_max) < 1e-4]
        n_rf_points = 512
        for col in ['NodeLabel', 'Node', 'NID']:
            if col in right_nodes.columns:
                n_rf_points = max(1, int(right_nodes[col].nunique()))
                break
        self.initialize_fixed_rf_points(n_rf_points=n_rf_points)

        for p in self.model.parameters():
            p.requires_grad_(True)
        for p in self.mat.parameters():
            p.requires_grad_(True)

        shift_params = list(self.mat.parameters())  # log_C1, log_C2, log_Ea_R
        adapt_log_vars = {}
        adapt_param_group = []
        if adaptive_loss_balance:
            # Keep stress term at a fixed explicit weight: it provides the most
            # direct gradient path to shift parameters (C1/C2/Ea_R).
            for name in ("data", "strain", "obs", "obs_rate", "rf", "pde"):
                p = nn.Parameter(torch.tensor(0.0, dtype=torch.float32, device=device))
                adapt_log_vars[name] = p
                adapt_param_group.append(p)

        opt_groups = [
            {'params': self.model.parameters(), 'lr': lr_network},
            {'params': shift_params,            'lr': lr_params},
        ]
        if len(adapt_param_group) > 0:
            opt_groups.append({'params': adapt_param_group, 'lr': min(1e-3, lr_params)})

        no_rf_mode = self.lambda_rf <= 0.0

        optimizer = Adam(opt_groups)
        scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)
        # Keep shift LR from collapsing too early; otherwise C1/C2/EaR freeze
        # in a wrong local minimum while the field network keeps fitting data.
        shift_lr_floor = max(6e-5, 0.25 * lr_params)
        net_lr_floor = max(1e-5, 0.05 * lr_network)
        # Adaptive-LR monitor state (plateau-triggered base-LR reduction).
        best_id_monitor = float('inf')
        plateau_count = 0
        plateau_patience = max(120, epochs // 25)
        id_improve_tol = 1e-3
        lr_reduce_net = 0.85
        lr_reduce_shift = 0.90
        lr_reduce_adapt = 0.85

        print(f"\nStarting EX4 inverse training ({epochs} epochs)...")
        print(f"  seq_length={seq_length}, batch_size={batch_size}")
        print(f"  adaptive_loss_balance={adaptive_loss_balance} adaptive_lr={adaptive_lr}")
        print(f"  Initial C1={self.mat.C1.item():.2f} (true {self.mat.TRUE_C1})")
        print(f"  Initial C2={self.mat.C2.item():.2f} (true {self.mat.TRUE_C2})")
        print(f"  Initial Ea_R={self.mat.Ea_R.item():.1f} (true {self.mat.TRUE_Ea_R})")
        print("-" * 80)

        def _clamp_shift_params():
            if hasattr(self.mat, 'log_C1'):
                self.mat.log_C1.data = torch.nan_to_num(
                    self.mat.log_C1.data, nan=math.log(10.0),
                    posinf=math.log(30.0), neginf=math.log(4.0)
                )
                self.mat.log_C1.data.clamp_(math.log(4.0), math.log(30.0))
            if hasattr(self.mat, 'log_C2'):
                self.mat.log_C2.data = torch.nan_to_num(
                    self.mat.log_C2.data, nan=math.log(30.0),
                    posinf=math.log(100.0), neginf=math.log(10.0)
                )
                self.mat.log_C2.data.clamp_(math.log(10.0), math.log(100.0))
            if hasattr(self.mat, 'log_Ea_R'):
                self.mat.log_Ea_R.data = torch.nan_to_num(
                    self.mat.log_Ea_R.data, nan=math.log(20000.0),
                    posinf=math.log(50000.0), neginf=math.log(3000.0)
                )
                self.mat.log_Ea_R.data.clamp_(math.log(3000.0), math.log(50000.0))

        def _sanitize_gradients(params):
            all_finite = True
            n_bad_tensors = 0
            for p in params:
                if p.grad is None:
                    continue
                finite = torch.isfinite(p.grad)
                if not finite.all():
                    all_finite = False
                    n_bad_tensors += 1
                    p.grad.data = torch.nan_to_num(p.grad.data, nan=0.0, posinf=0.0, neginf=0.0)
            return all_finite, n_bad_tensors

        # Staged identification schedule:
        #   warmup (network only) → C1/C2 phases → Ea/R phases → joint refine.
        # Network is never frozen after warmup; only targeted shift-parameter
        # freezing is used to reduce cross-coupling drift.
        phase1_end = max(200, epochs // 20)           # ~5% warmup
        phase2_end = max(phase1_end + 300, int(0.45 * epochs))  # wlf_only
        phase3_end = max(phase2_end + 300, int(0.68 * epochs))  # ear_only
        phase4_end = max(phase3_end + 200, int(0.82 * epochs))  # c1c2_refine
        phase5_end = max(phase4_end + 200, int(0.94 * epochs))  # ear_refine
        phase5_end = min(phase5_end, epochs - 1)

        print("  Phase schedule:")
        print(f"    field_warmup : [0, {phase1_end})")
        print(f"    wlf_only     : [{phase1_end}, {phase2_end})")
        print(f"    ear_only     : [{phase2_end}, {phase3_end})")
        print(f"    c1c2_refine  : [{phase3_end}, {phase4_end})")
        print(f"    ear_refine   : [{phase4_end}, {phase5_end})")
        print(f"    joint_finetune: [{phase5_end}, {epochs})")

        # Phase-wise best snapshots (observation-driven, no ground-truth usage):
        # - WLF-focused phases track best C1/C2 by cooling-window observables.
        # - Ea-focused phases track best Ea/R by recovery-window observables.
        best_wlf_score = float('inf')
        best_ear_score = float('inf')
        best_log_C1 = None
        best_log_C2 = None
        best_log_Ea = None
        prev_phase = None

        def _set_phase_requires_grad(epoch):
            # Field network stays trainable after warmup. Shift parameters are
            # selectively unfrozen by stage to improve identifiability.
            for p in self.model.parameters():
                p.requires_grad_(True)
            if epoch < phase1_end:
                phase = "field_warmup"
                if hasattr(self.mat, 'log_C1'):
                    self.mat.log_C1.requires_grad_(False)
                if hasattr(self.mat, 'log_C2'):
                    self.mat.log_C2.requires_grad_(False)
                if hasattr(self.mat, 'log_Ea_R'):
                    self.mat.log_Ea_R.requires_grad_(False)
            elif epoch < phase2_end:
                phase = "wlf_only"
                if hasattr(self.mat, 'log_C1'):
                    self.mat.log_C1.requires_grad_(True)
                if hasattr(self.mat, 'log_C2'):
                    self.mat.log_C2.requires_grad_(True)
                if hasattr(self.mat, 'log_Ea_R'):
                    self.mat.log_Ea_R.requires_grad_(False)
            elif epoch < phase3_end:
                phase = "ear_only"
                if hasattr(self.mat, 'log_C1'):
                    self.mat.log_C1.requires_grad_(False)
                if hasattr(self.mat, 'log_C2'):
                    self.mat.log_C2.requires_grad_(False)
                if hasattr(self.mat, 'log_Ea_R'):
                    self.mat.log_Ea_R.requires_grad_(True)
            elif epoch < phase4_end:
                phase = "c1c2_refine"
                if hasattr(self.mat, 'log_C1'):
                    self.mat.log_C1.requires_grad_(True)
                if hasattr(self.mat, 'log_C2'):
                    self.mat.log_C2.requires_grad_(True)
                if hasattr(self.mat, 'log_Ea_R'):
                    self.mat.log_Ea_R.requires_grad_(False)
            elif epoch < phase5_end:
                phase = "ear_refine"
                if hasattr(self.mat, 'log_C1'):
                    self.mat.log_C1.requires_grad_(False)
                if hasattr(self.mat, 'log_C2'):
                    self.mat.log_C2.requires_grad_(False)
                if hasattr(self.mat, 'log_Ea_R'):
                    self.mat.log_Ea_R.requires_grad_(True)
            else:
                phase = "joint_finetune"
                if hasattr(self.mat, 'log_C1'):
                    self.mat.log_C1.requires_grad_(True)
                if hasattr(self.mat, 'log_C2'):
                    self.mat.log_C2.requires_grad_(True)
                if hasattr(self.mat, 'log_Ea_R'):
                    self.mat.log_Ea_R.requires_grad_(True)
            return phase

        for epoch in range(epochs):
            phase = _set_phase_requires_grad(epoch)
            if prev_phase != phase:
                print(f"[Epoch {epoch:5d}] Phase transition: {prev_phase} → {phase}")
            prev_phase = phase
            self.model.train()
            optimizer.zero_grad()

            seq_data = self.sample_temporal_sequence(
                full_data,
                batch_size=batch_size,
                seq_length=seq_length,
                required_node_ids=self.rf_node_ids,
                node_id_col=self.rf_node_col,
            )

            coords_seq      = seq_data['coords_seq']
            u_data_seq      = seq_data['u_data_seq']
            strain_data_seq = seq_data['strain_data_seq']
            stress_data_seq = seq_data['stress_data_seq']
            sampled_times   = seq_data['times']
            T_seq           = seq_data['T_seq']
            dt_seq          = seq_data['dt_seq']
            N_pts = seq_data['N_pts']
            N_t   = seq_data['N_t']
            rf_indices      = seq_data['right_indices']
            left_indices    = seq_data['left_indices']
            yz_face_indices = seq_data['yz_face_indices']
            interior_start  = seq_data['interior_start']

            # ---- Single forward pass for all time steps ----
            strain_seq      = []
            u_pred_seq      = []
            coord_grads_seq = []

            for n in range(N_t):
                coords = coords_seq[n]
                t_val  = sampled_times[n]
                T_val  = T_seq[n]
                t_tensor = torch.full((N_pts, 1), t_val, dtype=torch.float32, device=device)
                T_tensor = torch.full((N_pts, 1), T_val, dtype=torch.float32, device=device)

                x_c = coords[:, 0:1].clone().requires_grad_(True)
                y_c = coords[:, 1:2].clone().requires_grad_(True)
                z_c = coords[:, 2:3].clone().requires_grad_(True)

                xh, yh, zh, th, Th = self.norm_coords(x_c, y_c, z_c, t_tensor, T_tensor)
                u_pred = self.model(xh, yh, zh, th, Th)
                strain = self.compute_strain(u_pred, x_c, y_c, z_c)

                u_pred_seq.append(u_pred)
                strain_seq.append(strain)
                coord_grads_seq.append((x_c, y_c, z_c))

            # ---- Rotate to material frame ----
            strain_mat_seq = []
            for n in range(N_t):
                strain_mat_seq.append(self.to_material_coords(strain_seq[n]))
            strain_mat_tensor = torch.stack(strain_mat_seq, dim=0)  # (N_t, N_pts, 6)

            # Temperature tensor for q recursion: (N_t, N_pts)
            T_for_q = torch.tensor(T_seq, dtype=torch.float32, device=device)  # (N_t,)
            T_for_q = T_for_q.unsqueeze(1).expand(N_t, N_pts)  # (N_t, N_pts)

            # ---- Q recursion with reduced time ----
            q_seq = self.compute_q_recursive(strain_mat_tensor, dt_seq, T_for_q, self.mat)

            rf_indices_torch = torch.tensor(rf_indices, dtype=torch.long, device=device)
            if rf_indices_torch.numel() == 0:
                print(f"[Epoch {epoch:5d}] RF node set is empty. right_indices={rf_indices}")
                break

            # ---- Accumulate losses ----
            loss_data_total     = torch.zeros((), device=device)
            loss_strain_total   = torch.zeros((), device=device)
            loss_stress_total   = torch.zeros((), device=device)
            loss_rf_total       = torch.zeros((), device=device)
            loss_obs_total      = torch.zeros((), device=device)
            loss_obs_rate_total = torch.zeros((), device=device)
            loss_bc_left_total  = torch.zeros((), device=device)
            loss_bc_right_total = torch.zeros((), device=device)
            loss_traction_total = torch.zeros((), device=device)
            loss_pde_total      = torch.zeros((), device=device)
            prev_u_obs_pred = None
            prev_u_obs_target = None
            prev_t_obs = None
            n_obs_rate_terms = 0
            # Observation-only monitors for phase-best snapshots
            rf_wlf_sum = 0.0
            rf_wlf_terms = 0
            obs_wlf_sum = 0.0
            obs_wlf_terms = 0
            rf_ear_sum = 0.0
            rf_ear_terms = 0
            obs_ear_sum = 0.0
            obs_ear_terms = 0
            obs_rate_ear_sum = 0.0
            obs_rate_ear_terms = 0
            pde_sample_interval = 30
            if phase in ("ear_only", "wlf_only", "c1c2_refine", "ear_refine"):
                pde_sample_interval = 15
            phase_rf_scale = 1.0
            if phase in ("ear_only", "ear_refine"):
                phase_rf_scale = 0.40 if not no_rf_mode else 0.25
            elif phase in ("wlf_only", "c1c2_refine"):
                phase_rf_scale = 1.20 if not no_rf_mode else 1.2
            # During parameter-identification phases, slightly downweight strain
            # to avoid it overwhelming RF/observable signals.
            phase_strain_scale = 1.0
            if phase in ("ear_only", "wlf_only"):
                phase_strain_scale = 0.50 if no_rf_mode else 0.35
            elif phase in ("c1c2_refine", "ear_refine"):
                phase_strain_scale = 0.42 if no_rf_mode else 0.30
            elif phase == "joint_finetune":
                phase_strain_scale = 0.60 if no_rf_mode else 0.40

            for n in range(N_t):
                coords   = coords_seq[n]
                u_data   = u_data_seq[n]
                u_pred   = u_pred_seq[n]
                strain_n = strain_seq[n]   # global frame
                q_n      = q_seq[n]        # material frame
                t_val = sampled_times[n]
                T_val = float(T_seq[n])
                # Temperature-window gating:
                # - WLF phases focus on high-T segment
                # - Ea phases focus on low-T segment
                phase_temp_scale = 1.0
                if phase in ("wlf_only", "c1c2_refine"):
                    if T_val >= 320.0:
                        phase_temp_scale = 1.6
                    elif T_val <= 315.0:
                        phase_temp_scale = 0.20
                    else:
                        phase_temp_scale = 0.90
                elif phase in ("ear_only", "ear_refine"):
                    if T_val <= 315.0:
                        phase_temp_scale = 1.8
                    elif T_val >= 320.0:
                        phase_temp_scale = 0.20
                    else:
                        phase_temp_scale = 0.90
                elif phase == "joint_finetune":
                    # Mild focus around cooling/high-T windows to protect C1/C2
                    # from drifting during late all-parameter updates.
                    if T_val >= 320.0:
                        phase_temp_scale = 1.15
                    elif T_val <= 310.0:
                        phase_temp_scale = 0.85
                    else:
                        phase_temp_scale = 1.0
                # Time-window gating (stronger than temperature-only gating):
                # - WLF phases: focus constrained cooling [50,100] s
                # - Ea phases : focus free-recovery [105,155] s
                phase_time_scale = 1.0
                if phase in ("wlf_only", "c1c2_refine"):
                    if 50.0 <= t_val < _T_CROSS_TIME:
                        phase_time_scale = 1.8   # WLF cooling zone: C1/C2 RF signal
                    elif _T_CROSS_TIME <= t_val < 100.0:
                        phase_time_scale = 0.40  # Arrhenius zone: suppress for C1/C2
                    elif t_val < 50.0:
                        phase_time_scale = 0.35
                    else:
                        phase_time_scale = 0.20
                elif phase in ("ear_only", "ear_refine"):
                    if t_val >= 105.0:
                        phase_time_scale = 1.8   # free recovery: main Ea/R signal
                    elif t_val >= 100.0:
                        phase_time_scale = 1.0   # spring-back
                    elif t_val >= _T_CROSS_TIME:
                        phase_time_scale = 1.5   # Arrhenius cooling (78-100s): Ea/R RF signal
                    elif t_val >= 50.0:
                        phase_time_scale = 0.20  # WLF cooling zone: suppress
                    else:
                        phase_time_scale = 0.10
                elif phase == "joint_finetune":
                    # Keep a light cooling emphasis in late stage so WLF params
                    # remain anchored by the most informative segment.
                    if 50.0 <= t_val <= 100.0:
                        phase_time_scale = 1.20
                    elif t_val < 50.0:
                        phase_time_scale = 0.75
                    elif t_val >= 105.0:
                        phase_time_scale = 1.05
                    else:
                        phase_time_scale = 0.90
                phase_focus_scale = phase_temp_scale * phase_time_scale

                # Data loss
                loss_data_total += phase_focus_scale * (torch.mean((u_pred - u_data) ** 2) / self.u_ref_sq)

                # Strain loss
                if strain_data_seq[n] is not None:
                    strain_fe = strain_data_seq[n]
                    eps_ref_sq = 0.15 ** 2  # max strain ≈15% for EX4
                    loss_strain_total += phase_focus_scale * torch.mean(
                        (strain_n - strain_fe) ** 2
                    ) / eps_ref_sq

                # Stress for RF / traction / PDE / stress-data loss
                strain_mat_n = strain_mat_seq[n]
                stress_n = self.compute_stress(strain_mat_n, q_n)

                # Stress-data loss — direct dense gradient path to shift parameters.
                # sigma_pred depends on q(a_T(C1,C2,Ea_R), strain), so matching
                # FE stress given matched strain uniquely constrains shift parameters.
                if self.lambda_stress > 0.0 and stress_data_seq[n] is not None:
                    sigma_ref_sq = 50.0 ** 2  # 50 MPa normalisation
                    loss_stress_total += torch.mean(
                        (stress_n - stress_data_seq[n]) ** 2
                    ) / sigma_ref_sq

                # PDE loss (sampled)
                compute_pde = (self.lambda_pde > 0.0) and (n % pde_sample_interval == 0) and (interior_start < N_pts)
                if compute_pde:
                    x_c_n, y_c_n, z_c_n = coord_grads_seq[n]
                    strain_pde = strain_seq[n][interior_start:]
                    q_pde      = q_n[interior_start:, :, :]
                    stress_pde = self.compute_stress(
                        self.to_material_coords(strain_pde), q_pde
                    )

                    s11 = stress_pde[:, 0:1]; s22 = stress_pde[:, 1:2]; s33 = stress_pde[:, 2:3]
                    s12 = stress_pde[:, 3:4]; s13 = stress_pde[:, 4:5]; s23 = stress_pde[:, 5:6]

                    def _pde_grad(s, leaf):
                        g = torch.autograd.grad(s, leaf, torch.ones_like(s),
                                                create_graph=True, allow_unused=True)[0]
                        return g[interior_start:] if g is not None else torch.zeros_like(s)

                    div_x = _pde_grad(s11, x_c_n) + _pde_grad(s12, y_c_n) + _pde_grad(s13, z_c_n)
                    div_y = _pde_grad(s12, x_c_n) + _pde_grad(s22, y_c_n) + _pde_grad(s23, z_c_n)
                    div_z = _pde_grad(s13, x_c_n) + _pde_grad(s23, y_c_n) + _pde_grad(s33, z_c_n)
                    pde_res = torch.cat([div_x, div_y, div_z], dim=1)

                    pde_ref_sq = (100.0 / self.Lx) ** 2
                    loss_pde_total += torch.mean(torch.sum(pde_res ** 2, dim=1)) / pde_ref_sq

                # RF loss (kept for t<100 with measured target; for t>=100 enforce
                # near-zero reaction on released end to couple recovery stage).
                if self.lambda_rf > 0.0:
                    rf_strain_n = strain_mat_seq[n][rf_indices_torch, :]
                    rf_q_n      = q_n[rf_indices_torch, :, :]
                    stress_rf   = self.compute_stress(rf_strain_n, rf_q_n)
                    rf_pred     = self.compute_reaction_force(stress_rf)

                    if t_val < 100.0:
                        rf_target = torch.tensor(
                            self.fe_loader.get_rf_by_time(t_val),
                            dtype=torch.float32, device=device
                        )
                        # Split cooling zone at T_cross crossing (~78.4s).
                        # WLF zone (t<78.4s, T>T_cross): C1/C2 sensitive.
                        # Arrhenius zone (t>=78.4s, T<T_cross): Ea/R sensitive.
                        if t_val < 50.0:
                            rf_time_scale = 0.60
                        elif t_val < _T_CROSS_TIME:  # WLF zone
                            if phase in ("wlf_only", "c1c2_refine"):
                                rf_time_scale = 2.0
                            elif phase in ("ear_only", "ear_refine"):
                                rf_time_scale = 0.20
                            else:
                                rf_time_scale = 1.2
                        else:  # Arrhenius zone — RF rises steeply (6→17 N), strong Ea/R signal
                            if phase in ("ear_only", "ear_refine"):
                                rf_time_scale = 3.0
                            elif phase in ("wlf_only", "c1c2_refine"):
                                rf_time_scale = 0.40
                            else:
                                rf_time_scale = 1.8
                    else:
                        rf_target = torch.zeros((), dtype=torch.float32, device=device)
                        if phase in ("ear_only", "ear_refine"):
                            rf_time_scale = 0.25 if t_val >= 105.0 else 0.15
                        else:
                            rf_time_scale = 0.12 if t_val >= 105.0 else 0.08

                    if (not torch.isfinite(rf_pred)) or (not torch.isfinite(rf_target)):
                        print(f"[Epoch {epoch:5d}] Non-finite RF branch at time {t_val:.6f}s")
                        print(f"  n_right={rf_indices_torch.numel()} rf_target={rf_target.item()}")
                        print(f"  finite(rf_strain)={torch.isfinite(rf_strain_n).all().item()} "
                              f"finite(rf_q)={torch.isfinite(rf_q_n).all().item()} "
                              f"finite(stress_rf)={torch.isfinite(stress_rf).all().item()}")
                        if rf_indices_torch.numel() > 0:
                            print(f"  stress_rf range: "
                                  f"[{torch.nan_to_num(stress_rf).min().item():.4e}, "
                                  f"{torch.nan_to_num(stress_rf).max().item():.4e}]")
                        loss_rf_total = torch.tensor(float('nan'), device=device)
                        break
                    rf_denom = torch.clamp(torch.abs(rf_target), min=self.rf_rel_floor)
                    rf_rel_sq = ((rf_pred - rf_target) / rf_denom) ** 2
                    loss_rf_total += (rf_time_scale * phase_rf_scale * phase_focus_scale) * rf_rel_sq
                    rf_rel_sq_val = float(torch.clamp(rf_rel_sq.detach(), max=1e6).item())
                    if 50.0 <= t_val < _T_CROSS_TIME:
                        rf_wlf_sum += rf_rel_sq_val
                        rf_wlf_terms += 1
                    if (t_val >= _T_CROSS_TIME and t_val < 100.0) or (t_val >= 105.0):
                        rf_ear_sum += rf_rel_sq_val
                        rf_ear_terms += 1

                # Observable loss on loaded-end displacement history.
                # This is the most informative supervision during unloading/recovery,
                # where RF ceases to be meaningful but shift parameters still control
                # residual deformation and recovery kinetics.
                u_obs_pred = u_pred_seq[n][rf_indices_torch, 0].mean()
                u_obs_target = torch.tensor(
                    self.fe_loader.get_u_by_time(t_val),
                    dtype=torch.float32, device=device
                )
                obs_weight = 1.0
                if t_val >= 100.0:
                    obs_weight = 5.0
                elif t_val >= 50.0:
                    obs_weight = 2.0
                if abs(t_val - 100.0) <= 5.0 or abs(t_val - 105.0) <= 5.0 or abs(t_val - 154.57) <= 5.0:
                    obs_weight *= 2.0
                # In no-RF mode, emphasize phase-sensitive time windows to improve
                # identifiability separation:
                # - WLF phases focus on cooling/constrained segment (50~100 s)
                # - Ea phases focus on free recovery segment (>=105 s)
                phase_obs_scale = 1.0
                if phase in ("wlf_only", "c1c2_refine"):
                    if 50.0 <= t_val <= 100.0:
                        phase_obs_scale = 3.0 if no_rf_mode else 2.0
                    elif t_val < 50.0:
                        phase_obs_scale = 0.35 if no_rf_mode else 0.70
                    else:
                        phase_obs_scale = 0.50 if no_rf_mode else 0.80
                elif phase in ("ear_only", "ear_refine"):
                    if t_val >= 105.0:
                        phase_obs_scale = 3.0 if no_rf_mode else 2.5
                    elif t_val >= 50.0:
                        phase_obs_scale = 1.2 if no_rf_mode else 0.9
                    else:
                        phase_obs_scale = 0.30 if no_rf_mode else 0.60
                obs_rel_sq = ((u_obs_pred - u_obs_target) ** 2) / self.u_ref_sq
                loss_obs_total += (obs_weight * phase_obs_scale * phase_focus_scale) * obs_rel_sq
                obs_rel_sq_val = float(torch.clamp(obs_rel_sq.detach(), max=1e6).item())
                if 50.0 <= t_val < 100.0:
                    obs_wlf_sum += obs_rel_sq_val
                    obs_wlf_terms += 1
                if t_val >= 105.0:
                    obs_ear_sum += obs_rel_sq_val
                    obs_ear_terms += 1

                # Observable-rate loss (du/dt) on loaded-end displacement history.
                # This term is especially informative for Ea/R during free recovery.
                if prev_u_obs_pred is not None and prev_t_obs is not None:
                    dt_obs = max(t_val - prev_t_obs, 1e-6)
                    u_rate_pred = (u_obs_pred - prev_u_obs_pred) / dt_obs
                    u_rate_target = (u_obs_target - prev_u_obs_target) / dt_obs
                    # Only apply rate supervision near unloading/recovery.
                    obs_rate_scale = 0.0
                    if t_val >= 105.0:
                        obs_rate_scale = 1.0
                    elif t_val >= 100.0:
                        obs_rate_scale = 0.5
                    if phase in ("ear_only", "ear_refine"):
                        obs_rate_scale *= 1.10
                    elif phase in ("wlf_only", "c1c2_refine"):
                        obs_rate_scale *= 0.30
                    if obs_rate_scale > 0.0:
                        # Relative rate error with floor avoids scale explosion.
                        rate_denom = torch.clamp(torch.abs(u_rate_target), min=0.08)
                        rate_res_sq = torch.clamp(((u_rate_pred - u_rate_target) / rate_denom) ** 2, max=25.0)
                        loss_obs_rate_total += (obs_rate_scale * phase_focus_scale) * rate_res_sq
                        if t_val >= 105.0:
                            obs_rate_ear_sum += float(torch.clamp(rate_res_sq.detach(), max=1e6).item())
                            obs_rate_ear_terms += 1
                        n_obs_rate_terms += 1

                prev_u_obs_pred = u_obs_pred
                prev_u_obs_target = u_obs_target
                prev_t_obs = t_val

                # BC right (prescribed displacement)
                if t_val < 100.0:
                    u_bc_right_pred   = u_pred_seq[n][rf_indices_torch, :]
                    u_bc_right_target = u_data_seq[n][rf_indices_torch, :]
                    loss_bc_right_total += torch.mean(
                        (u_bc_right_pred - u_bc_right_target) ** 2
                    ) / self.u_ref_sq
                else:
                    # After unloading (t >= 100 s), the right end is released.
                    # Enforce traction-free condition on the right face to keep
                    # recovery-stage mechanics coupled to shift parameters.
                    stress_right = stress_n[rf_indices_torch, :]
                    tx = torch.stack(
                        [stress_right[:, 0], stress_right[:, 3], stress_right[:, 4]],
                        dim=1
                    )  # traction components on x-normal plane
                    sigma_ref_sq = 100.0 ** 2
                    free_w = 2.0 if t_val >= 105.0 else 1.0
                    loss_bc_right_total += free_w * torch.mean(tx ** 2) / sigma_ref_sq

                # BC left (fixed, u=0)
                if len(left_indices) > 0:
                    li = torch.tensor(left_indices, dtype=torch.long, device=device)
                    loss_bc_left_total += torch.mean(
                        u_pred_seq[n][li, :] ** 2
                    ) / self.u_ref_sq

                # Traction-free lateral BC
                if len(yz_face_indices) > 0:
                    yi = torch.tensor(yz_face_indices, dtype=torch.long, device=device)
                    coords_yz = coords[yi, :]
                    stress_yz = stress_n[yi, :]
                    sigma_ref_sq = 100.0 ** 2
                    y_min = self.bounds['y_min']; y_max = self.bounds['y_max']
                    z_min = self.bounds['z_min']; z_max = self.bounds['z_max']
                    tol = 1e-4
                    is_y = (torch.abs(coords_yz[:, 1] - y_min) < tol) | \
                           (torch.abs(coords_yz[:, 1] - y_max) < tol)
                    is_z = (torch.abs(coords_yz[:, 2] - z_min) < tol) | \
                           (torch.abs(coords_yz[:, 2] - z_max) < tol)
                    if is_y.any():
                        sy = stress_yz[is_y]
                        ty = torch.stack([sy[:, 3], sy[:, 1], sy[:, 5]], dim=1)
                        loss_traction_total += torch.mean(ty ** 2) / sigma_ref_sq
                    if is_z.any():
                        sz = stress_yz[is_z]
                        tz = torch.stack([sz[:, 4], sz[:, 5], sz[:, 2]], dim=1)
                        loss_traction_total += torch.mean(tz ** 2) / sigma_ref_sq

            # Average over time steps
            loss_data     = loss_data_total     / N_t
            loss_strain   = phase_strain_scale * (loss_strain_total / N_t)
            loss_stress   = loss_stress_total   / N_t
            loss_rf       = loss_rf_total       / N_t
            loss_obs      = loss_obs_total      / N_t
            loss_obs_rate = loss_obs_rate_total / max(1, n_obs_rate_terms)
            loss_bc_left  = loss_bc_left_total  / N_t
            loss_bc_right = loss_bc_right_total / N_t
            loss_traction = loss_traction_total / N_t
            n_pde_computed = max(1, sum(1 for i in range(N_t)
                                        if i % pde_sample_interval == 0))
            loss_pde = loss_pde_total / n_pde_computed

            loss_penalty = self.mat.compute_parameter_penalty()
            # Phase-specific identification monitors (observation windows only).
            wlf_monitor = float('inf')
            if rf_wlf_terms > 0 or obs_wlf_terms > 0:
                wlf_monitor = 0.0
                if rf_wlf_terms > 0:
                    wlf_monitor += rf_wlf_sum / rf_wlf_terms
                if obs_wlf_terms > 0:
                    wlf_monitor += 0.25 * (obs_wlf_sum / obs_wlf_terms)
            ear_monitor = float('inf')
            if rf_ear_terms > 0 or obs_ear_terms > 0 or obs_rate_ear_terms > 0:
                ear_monitor = 0.0
                if obs_ear_terms > 0:
                    ear_monitor += obs_ear_sum / obs_ear_terms
                if obs_rate_ear_terms > 0:
                    ear_monitor += 0.80 * (obs_rate_ear_sum / obs_rate_ear_terms)
                if rf_ear_terms > 0:
                    ear_monitor += 0.20 * (rf_ear_sum / rf_ear_terms)
            # Update best-parameter snapshots (observation-window monitor).
            if phase in ("wlf_only", "c1c2_refine", "joint_finetune") and np.isfinite(wlf_monitor):
                if wlf_monitor < best_wlf_score:
                    best_wlf_score = wlf_monitor
                    best_log_C1 = self.mat.log_C1.detach().clone()
                    best_log_C2 = self.mat.log_C2.detach().clone()
            if phase in ("ear_only", "ear_refine", "joint_finetune") and np.isfinite(ear_monitor):
                if ear_monitor < best_ear_score:
                    best_ear_score = ear_monitor
                    best_log_Ea = self.mat.log_Ea_R.detach().clone()

            # Uniform loss weights — with stress-data loss providing the primary
            # shift-parameter gradient, phase-specific reweighting is not needed.
            phase_w_data = 1.0
            phase_w_strain = 1.0
            phase_w_obs = 1.0
            phase_w_obs_rate = 1.0
            phase_w_rf = 1.0
            phase_w_pde = 1.0

            base_terms = {
                'data':     (phase_w_data * self.lambda_data) * loss_data,
                'strain':   (phase_w_strain * self.lambda_strain) * loss_strain,
                'obs':      (phase_w_obs * self.lambda_obs) * loss_obs,
                'obs_rate': (phase_w_obs_rate * self.lambda_obs_rate) * loss_obs_rate,
                'rf':       (phase_w_rf * self.lambda_rf) * loss_rf,
                'pde':      (phase_w_pde * self.lambda_pde) * loss_pde,
            }
            loss_anchor = torch.zeros((), device=device)

            loss_total = (
                self.lambda_bc_left  * loss_bc_left +
                self.lambda_bc_right * loss_bc_right +
                self.lambda_traction * loss_traction +
                self.lambda_stress * loss_stress +
                50.0 * loss_penalty +
                loss_anchor
            )
            if adaptive_loss_balance and len(adapt_log_vars) > 0:
                for name, term in base_terms.items():
                    coeff = float(term.detach().abs().item())
                    if coeff <= 0.0:
                        loss_total = loss_total + term
                        continue
                    lv = adapt_log_vars[name]
                    # Heteroscedastic uncertainty weighting:
                    # exp(-lv) * L + reg(lv). This auto-downweights exploding terms.
                    loss_total = loss_total + torch.exp(-lv) * term + 0.05 * (lv ** 2)
            else:
                for term in base_terms.values():
                    loss_total = loss_total + term

            if not torch.isfinite(loss_total):
                print(f"[Epoch {epoch:5d}] Non-finite loss detected.")
                print(f"  finite(data)={torch.isfinite(loss_data).item()} "
                      f"finite(strain)={torch.isfinite(loss_strain).item()} "
                      f"finite(rf)={torch.isfinite(loss_rf).item()} "
                      f"finite(obs)={torch.isfinite(loss_obs).item()} "
                      f"finite(pde)={torch.isfinite(loss_pde).item()} "
                      f"finite(penalty)={torch.isfinite(loss_penalty).item()}")
                print(f"  C1={self.mat.C1.item()} C2={self.mat.C2.item()} Ea_R={self.mat.Ea_R.item()}")
                break

            loss_total.backward()

            # In warmup phase, zero any shift-parameter gradients that leaked
            # through requires_grad_(False) (safety guard only).
            if phase == "field_warmup":
                for attr in ('log_C1', 'log_C2', 'log_Ea_R'):
                    p = getattr(self.mat, attr, None)
                    if p is not None and p.grad is not None:
                        p.grad.zero_()

            c1_bad = int(hasattr(self.mat, 'log_C1') and self.mat.log_C1.grad is not None
                         and (not torch.isfinite(self.mat.log_C1.grad).all()))
            c2_bad = int(hasattr(self.mat, 'log_C2') and self.mat.log_C2.grad is not None
                         and (not torch.isfinite(self.mat.log_C2.grad).all()))
            ear_bad = int(hasattr(self.mat, 'log_Ea_R') and self.mat.log_Ea_R.grad is not None
                          and (not torch.isfinite(self.mat.log_Ea_R.grad).all()))

            grads_finite_model, n_bad_model = _sanitize_gradients(self.model.parameters())
            grads_finite_shift, n_bad_shift = _sanitize_gradients(shift_params)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            torch.nn.utils.clip_grad_norm_(shift_params, max_norm=0.8)

            if not (grads_finite_model and grads_finite_shift):
                print(f"[Epoch {epoch:5d}] Non-finite gradients detected; sanitizing and continuing.")
                print(f"  bad_model_tensors={n_bad_model} bad_shift_tensors={n_bad_shift} "
                      f"bad_logC1={c1_bad} bad_logC2={c2_bad} bad_logEaR={ear_bad}")

            optimizer.step()
            scheduler.step()
            if adaptive_loss_balance and len(adapt_log_vars) > 0:
                for p in adapt_log_vars.values():
                    p.data.clamp_(-3.0, 3.0)
            # Enforce a practical lower bound for shift-parameter LR.
            if optimizer.param_groups[1]['lr'] < shift_lr_floor:
                optimizer.param_groups[1]['lr'] = shift_lr_floor
            if optimizer.param_groups[0]['lr'] < net_lr_floor:
                optimizer.param_groups[0]['lr'] = net_lr_floor

            if adaptive_lr:
                # Monitor only active identification channels. In particular,
                # exclude PDE when lambda_pde=0 to avoid irrelevant LR decay.
                id_monitor = (
                    loss_obs.detach().item() +
                    0.6 * loss_obs_rate.detach().item()
                )
                if self.lambda_stress > 0.0:
                    id_monitor += 1.0 * loss_stress.detach().item()
                if self.lambda_rf > 0.0:
                    id_monitor += 0.6 * loss_rf.detach().item()
                if self.lambda_pde > 0.0:
                    id_monitor += 0.4 * loss_pde.detach().item()
                if id_monitor < best_id_monitor * (1.0 - id_improve_tol):
                    best_id_monitor = id_monitor
                    plateau_count = 0
                else:
                    plateau_count += 1
                    if plateau_count >= plateau_patience:
                        # Reduce cosine base_lrs so future scheduler values also drop.
                        scheduler.base_lrs[0] = max(net_lr_floor, scheduler.base_lrs[0] * lr_reduce_net)
                        scheduler.base_lrs[1] = max(shift_lr_floor, scheduler.base_lrs[1] * lr_reduce_shift)
                        if len(scheduler.base_lrs) > 2:
                            scheduler.base_lrs[2] = max(1e-5, scheduler.base_lrs[2] * lr_reduce_adapt)
                        optimizer.param_groups[0]['lr'] = max(net_lr_floor, optimizer.param_groups[0]['lr'] * lr_reduce_net)
                        optimizer.param_groups[1]['lr'] = max(shift_lr_floor, optimizer.param_groups[1]['lr'] * lr_reduce_shift)
                        if len(optimizer.param_groups) > 2:
                            optimizer.param_groups[2]['lr'] = max(1e-5, optimizer.param_groups[2]['lr'] * lr_reduce_adapt)
                        plateau_count = 0
                        print(
                            f"[Epoch {epoch:5d}] adaptive-lr reduce: "
                            f"net={optimizer.param_groups[0]['lr']:.2e} "
                            f"shift={optimizer.param_groups[1]['lr']:.2e} "
                            f"id_monitor={id_monitor:.4e}"
                        )
            _clamp_shift_params()

            # Store history
            self.loss_history.append(loss_total.item())
            self.loss_components_history.append({
                'data':     loss_data.item(),
                'strain':   loss_strain.item(),
                'stress':   loss_stress.item(),
                'obs':      loss_obs.item(),
                'obs_rate': loss_obs_rate.item(),
                'bc_left':  loss_bc_left.item(),
                'bc_right': loss_bc_right.item(),
                'traction': loss_traction.item(),
                'rf':       loss_rf.item(),
                'pde':      loss_pde.item(),
                'penalty':  loss_penalty.item(),
                'anchor':   loss_anchor.item(),
            })
            self.param_history.append({
                'C1':   self.mat.C1.item(),
                'C2':   self.mat.C2.item(),
                'Ea_R': self.mat.Ea_R.item(),
            })

            if epoch % log_interval == 0 or epoch == epochs - 1:
                C1_now   = self.mat.C1.item()
                C2_now   = self.mat.C2.item()
                EaR_now  = self.mat.Ea_R.item()
                lr_net   = optimizer.param_groups[0]['lr']
                lr_sft   = optimizer.param_groups[1]['lr']
                t_range  = f"[{sampled_times[0]:.1f},{sampled_times[-1]:.1f}]s"
                T_min_seq = min(T_seq)
                T_max_seq = max(T_seq)
                T_range  = f"[{T_min_seq:.0f},{T_max_seq:.0f}]K"
                print(
                    f"[Epoch {epoch:5d}] total={loss_total.item():.4e} | "
                    f"data={loss_data.item():.4e} strain={loss_strain.item():.4e} "
                    f"stress={loss_stress.item():.4e} "
                    f"obs={loss_obs.item():.4e} obs_rate={loss_obs_rate.item():.4e} "
                    + (f"rf={loss_rf.item():.4e} " if self.lambda_rf > 0.0 else "")
                    + f"pde={loss_pde.item():.4e}"
                )
                if adaptive_loss_balance and len(adapt_log_vars) > 0:
                    w_line = " ".join(
                        f"{k}={torch.exp(-v.detach()).item():.2f}"
                        for k, v in adapt_log_vars.items()
                    )
                    print(f"  [AdaptiveW] {w_line}")
                if np.isfinite(wlf_monitor) or np.isfinite(ear_monitor):
                    wlf_str = f"{wlf_monitor:.4e}" if np.isfinite(wlf_monitor) else "nan"
                    ear_str = f"{ear_monitor:.4e}" if np.isfinite(ear_monitor) else "nan"
                    print(f"  [IDMonitor] wlf={wlf_str} ear={ear_str} best_wlf={best_wlf_score:.4e} best_ear={best_ear_score:.4e}")
                print(
                    f"  [Phase] {phase}")
                print(
                    f"  [Shift] C1={C1_now:.3f}(T:{self.mat.TRUE_C1})  "
                    f"C2={C2_now:.2f}(T:{self.mat.TRUE_C2})  "
                    f"Ea_R={EaR_now:.0f}(T:{self.mat.TRUE_Ea_R})"
                )
                print(f"  [LR] net={lr_net:.2e} shift={lr_sft:.2e}  "
                      f"t={t_range} T={T_range} N_t={N_t}")

        # Restore phase-best snapshots to prevent late-stage drift away from the
        # most informative windows for each parameter subset.
        with torch.no_grad():
            restored = []
            if best_log_C1 is not None and hasattr(self.mat, 'log_C1'):
                self.mat.log_C1.copy_(best_log_C1)
                restored.append("C1")
            if best_log_C2 is not None and hasattr(self.mat, 'log_C2'):
                self.mat.log_C2.copy_(best_log_C2)
                restored.append("C2")
            if best_log_Ea is not None and hasattr(self.mat, 'log_Ea_R'):
                self.mat.log_Ea_R.copy_(best_log_Ea)
                restored.append("Ea_R")
            _clamp_shift_params()
        if len(restored) > 0:
            print(f"Restored phase-best shift params: {', '.join(restored)}")
            if np.isfinite(best_wlf_score) or np.isfinite(best_ear_score):
                wlf_str = f"{best_wlf_score:.4e}" if np.isfinite(best_wlf_score) else "nan"
                ear_str = f"{best_ear_score:.4e}" if np.isfinite(best_ear_score) else "nan"
                print(f"  best monitors: wlf={wlf_str}, ear={ear_str}")

        print(f"\nTraining complete: {time.time() - start_time:.1f}s")

    # ------------------------------------------------------------------
    # Persistence helpers
    # ------------------------------------------------------------------
    def save_loss_history(self, path):
        rows = []
        for i, (total, comps) in enumerate(
                zip(self.loss_history, self.loss_components_history)):
            row = {'epoch': i, 'loss_total': total}
            row.update(comps)
            rows.append(row)
        pd.DataFrame(rows).to_csv(path, index=False)
        print(f"Loss history saved to {path}")

    def save_param_history(self, path):
        rows = [{'epoch': i, **h} for i, h in enumerate(self.param_history)]
        pd.DataFrame(rows).to_csv(path, index=False)
        print(f"Parameter history saved to {path}")


# ---------------------------------------------------------------------------
# 5. Visualisation helpers
# ---------------------------------------------------------------------------

def plot_training_history(solver, save_dir):
    plt.rcParams['font.family'] = 'Times New Roman'
    plt.rcParams['font.size'] = 12

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    epochs = range(len(solver.loss_history))

    ax = axes[0]
    ax.semilogy(epochs, solver.loss_history, 'k-', linewidth=1.5, label='Total')
    if solver.loss_components_history:
        for key, col, ls in [
            ('data', 'C0', '-'), ('strain', 'C1', '-'),
            ('stress', 'C8', '-'),
            ('obs', 'C2', '-'), ('obs_rate', 'C3', '-.'),
            ('rf', 'C4', '-'), ('pde', 'C5', '--'),
            ('bc_left', 'C6', ':'), ('bc_right', 'C7', ':'),
        ]:
            # LSTM variants do not track every component key (e.g., stress/obs).
            if not any(key in h for h in solver.loss_components_history):
                continue
            vals = [h.get(key, 0.0) for h in solver.loss_components_history]
            ax.semilogy(epochs, vals, color=col, linestyle=ls,
                        linewidth=1, label=key, alpha=0.7)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.set_title('Training Loss')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3, which='both')

    ax = axes[1]
    if solver.param_history:
        ep = range(len(solver.param_history))
        C1_true  = InverseShiftParams.TRUE_C1
        C2_true  = InverseShiftParams.TRUE_C2
        EaR_true = InverseShiftParams.TRUE_Ea_R

        ax2 = ax.twinx()
        l1, = ax.plot(ep,  [h['C1']   for h in solver.param_history],
                      'C0-', linewidth=1.5, label=f'C1 (true={C1_true})')
        l2, = ax.plot(ep,  [h['C2']   for h in solver.param_history],
                      'C1-', linewidth=1.5, label=f'C2 (true={C2_true})')
        ax.axhline(C1_true, color='C0', linestyle='--', alpha=0.5)
        ax.axhline(C2_true, color='C1', linestyle='--', alpha=0.5)
        l3, = ax2.semilogy(ep, [h['Ea_R'] for h in solver.param_history],
                           'C2-', linewidth=1.5, label=f'Ea/R (true={EaR_true})')
        ax2.axhline(EaR_true, color='C2', linestyle='--', alpha=0.5)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('C1, C2')
        ax2.set_ylabel('Ea/R [K]')
        ax.set_title('Shift Parameter Convergence')
        ax.legend(handles=[l1, l2, l3], fontsize=9)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    save_path = Path(save_dir) / 'ex4_training_history.png'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"Training history saved to {save_path}")
    plt.close()


# ---------------------------------------------------------------------------
# 6. Main
# ---------------------------------------------------------------------------


class LSTMInverseEX4(nn.Module):
    """
    LSTM-PINN for EX4 inverse shift parameter identification.

    Input sequence per time step:
      Spatial features from (x_hat, y_hat, z_hat, x_local, y_local, T_hat) → MLP → features
    LSTM processes the temporal sequence, decoder maps to (u1, u2, u3).

    Args:
        spatial_hidden: Hidden sizes for the spatial MLP encoder
        lstm_hidden:    LSTM hidden state size
        lstm_layers:    Number of LSTM layers
        dropout:        Dropout probability
    """

    def __init__(self, spatial_hidden=(64, 64), lstm_hidden=64,
                 lstm_layers=2, dropout=0.0):
        super().__init__()

        self.lstm_hidden = lstm_hidden
        self.lstm_layers = lstm_layers

        c = math.cos(math.radians(45.0))
        s = math.sin(math.radians(45.0))
        self.register_buffer("cos_theta", torch.tensor(c, dtype=torch.float32))
        self.register_buffer("sin_theta", torch.tensor(s, dtype=torch.float32))

        # Spatial encoder: 6D (x,y,z,x_local,y_local,T) → features
        dims = [6, *spatial_hidden]
        mlp = []
        for i in range(len(dims) - 1):
            mlp.append(nn.Linear(dims[i], dims[i + 1]))
            mlp.append(nn.Tanh())
            if dropout > 0:
                mlp.append(nn.Dropout(dropout))
        self.spatial_encoder = nn.Sequential(*mlp)

        # LSTM
        self.lstm = nn.LSTM(
            input_size=dims[-1],
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            dropout=dropout if lstm_layers > 1 else 0.0,
        )

        # Decoder: hidden → displacement (u1,u2,u3)
        self.decoder = nn.Sequential(
            nn.Linear(lstm_hidden, lstm_hidden),
            nn.Tanh(),
            nn.Linear(lstm_hidden, 3),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def _encode_spatial(self, x, y, z, T):
        """
        Encode (x,y,z, fiber-local, T) to spatial features.

        Args:
            x, y, z, T: shape (batch, seq, 1) or (batch, 1)
        Returns:
            features: same leading dims → (batch, seq, feature_dim)
        """
        x_local = x * self.cos_theta + y * self.sin_theta
        y_local = -x * self.sin_theta + y * self.cos_theta
        h = torch.cat([x, y, z, x_local, y_local, T], dim=-1)
        return self.spatial_encoder(h)

    def forward_sequence(self, x, y, z, T, h0=None, c0=None):
        """
        Process a temporal sequence.

        Args:
            x, y, z, T: Normalised coords, shape (batch, seq, 1)
            h0, c0:     Optional initial LSTM hidden state

        Returns:
            u_seq:    Displacements (batch, seq, 3)
            (h, c):  Final hidden state
        """
        spatial_feats = self._encode_spatial(x, y, z, T)  # (B, S, F)
        lstm_out, (h, c) = self.lstm(
            spatial_feats,
            (h0, c0) if h0 is not None else None
        )  # (B, S, lstm_hidden)
        u_seq = self.decoder(lstm_out)  # (B, S, 3)
        return u_seq, (h, c)

    def forward_step(self, x, y, z, T, h, c):
        """
        Process ONE time step with given hidden state.

        Used in the training loop to make per-step strain computation O(1) instead
        of O(N_t) per step (which was the source of the 24-hour hang: full-sequence
        autograd.grad through N_t LSTM steps, called N_t times = O(N_t²) total).

        Args:
            x, y, z, T: (batch, 1) — spatial coords and temperature for this step
            h, c:       (lstm_layers, batch, lstm_hidden) — LSTM hidden state

        Returns:
            u:        (batch, 3) displacement at this step
            (h, c):  Updated hidden state
        """
        # unsqueeze seq dimension → (batch, 1, feat)
        feats = self._encode_spatial(
            x.unsqueeze(1), y.unsqueeze(1), z.unsqueeze(1), T.unsqueeze(1)
        )
        out, (h_new, c_new) = self.lstm(feats, (h, c))   # out: (batch, 1, H)
        u = self.decoder(out[:, 0, :])                    # (batch, 3)
        return u, (h_new, c_new)


# ---------------------------------------------------------------------------
# LSTM Inverse Solver — inherits physics from InversePINNSolver
# ---------------------------------------------------------------------------

class InverseLSTMSolverEX4(InversePINNSolver):
    """
    LSTM-PINN solver for EX4 inverse shift parameter identification.

    Overrides train() to use LSTM sequence processing.
    Inherits compute_q_recursive, compute_stress, shift_factor physics
    from InversePINNSolver.
    """

    @staticmethod
    def _compute_strain_from_step(u_n, x_n, y_n, z_n):
        """
        Compute engineering strain at one time step from per-step leaf tensors.

        Cost: ONE local LSTM-step backward (O(1)), regardless of sequence length.

        Args:
            u_n:  (N_pts, 3) — displacement output from model.forward_step
            x_n, y_n, z_n: (N_pts, 1) leaf tensors used to call forward_step
        """
        u1, u2, u3 = u_n[:, 0:1], u_n[:, 1:2], u_n[:, 2:3]

        def _g(ui, coord):
            g = torch.autograd.grad(
                ui, coord, torch.ones_like(ui),
                create_graph=True, retain_graph=True, allow_unused=True
            )[0]
            return g if g is not None else torch.zeros_like(coord)

        e11 = _g(u1, x_n)
        e22 = _g(u2, y_n)
        e33 = _g(u3, z_n)
        g12 = _g(u1, y_n) + _g(u2, x_n)
        g13 = _g(u1, z_n) + _g(u3, x_n)
        g23 = _g(u2, z_n) + _g(u3, y_n)
        return torch.cat([e11, e22, e33, g12, g13, g23], dim=1)

    # Legacy method retained for reference — DO NOT CALL in training loops.
    # Calling this N_t times inside a loop produces O(N_t²) BPTT passes because
    # each call backpropagates through the FULL sequence (x_seq appears at all N_t
    # LSTM input positions). With N_t=56 and 9 grad calls per step: 504 full BPTT
    # passes per epoch → ~25 s/epoch on CPU → 69 hours for 10k epochs.
    @staticmethod
    def _compute_strain_from_seq(u_seq, x_seq, y_seq, z_seq, n, retain_graph=True):
        raise RuntimeError(
            "_compute_strain_from_seq causes O(N_t²) BPTT and has been disabled. "
            "Use _compute_strain_from_step with forward_step instead."
        )

    def train(self, epochs=10000, batch_size=64,
              lr_network=2e-3, lr_params=1e-2,
              log_interval=100, seq_length=24):

        start_time = time.time()
        full_data = self.fe_loader.full_data

        # Displacement reference scale
        x_max = self.bounds['x_max']
        right_data = full_data[np.abs(full_data['X'].values - x_max) < 1e-4]
        u_mag = np.linalg.norm(right_data[['U1', 'U2', 'U3']].values, axis=1)
        u_ref = float(np.max(u_mag))
        self.u_ref_sq = (u_ref + 1e-10) ** 2

        rf_values = torch.tensor(self.fe_loader.rf_data['RF'].values,
                                  dtype=torch.float32, device=device)
        rf_max_abs = float(torch.max(torch.abs(rf_values)).item()) if rf_values.numel() > 0 else 0.0
        rf_floor_val = max(2.0, 0.15 * rf_max_abs)
        self.rf_rel_floor = torch.tensor(rf_floor_val, dtype=torch.float32, device=device)
        print(f"\nU scale (max|u|): {u_ref:.4f} mm")
        print(f"RF range: [{rf_values.min().item():.2f}, {rf_values.max().item():.2f}] N")

        # Align with EX2: use all available right-face nodes for RF supervision.
        right_nodes = full_data[np.abs(full_data['X'].values - x_max) < 1e-4]
        n_rf_points = 512
        for col in ['NodeLabel', 'Node', 'NID']:
            if col in right_nodes.columns:
                n_rf_points = max(1, int(right_nodes[col].nunique()))
                break
        self.initialize_fixed_rf_points(n_rf_points=n_rf_points)

        for p in self.model.parameters():
            p.requires_grad_(True)
        for p in self.mat.parameters():
            p.requires_grad_(True)

        shift_params = list(self.mat.parameters())

        optimizer = Adam([
            {'params': self.model.parameters(), 'lr': lr_network},
            {'params': shift_params,            'lr': lr_params},
        ])
        scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)

        print(f"\nStarting EX4 LSTM inverse training ({epochs} epochs)...")
        print(f"  seq_length={seq_length}, batch_size={batch_size}")
        print(f"  Initial C1={self.mat.C1.item():.2f} (true {self.mat.TRUE_C1})")
        print(f"  Initial C2={self.mat.C2.item():.2f} (true {self.mat.TRUE_C2})")
        print(f"  Initial Ea_R={self.mat.Ea_R.item():.1f} (true {self.mat.TRUE_Ea_R})")
        print("-" * 80)

        for epoch in range(epochs):
            self.model.train()
            optimizer.zero_grad()

            seq_data = self.sample_temporal_sequence(
                full_data,
                batch_size=batch_size,
                seq_length=seq_length,
                required_node_ids=self.rf_node_ids,
                node_id_col=self.rf_node_col,
            )

            coords_seq      = seq_data['coords_seq']
            u_data_seq      = seq_data['u_data_seq']
            strain_data_seq = seq_data['strain_data_seq']
            sampled_times   = seq_data['times']
            T_seq           = seq_data['T_seq']
            dt_seq          = seq_data['dt_seq']
            N_pts = seq_data['N_pts']
            N_t   = seq_data['N_t']
            rf_indices      = seq_data['right_indices']
            left_indices    = seq_data['left_indices']
            yz_face_indices = seq_data['yz_face_indices']
            interior_start  = seq_data['interior_start']

            # ---- Spatial coordinates (fixed across time for each node) ----
            x_arr = coords_seq[0][:, 0:1].clone()  # (N_pts, 1)
            y_arr = coords_seq[0][:, 1:2].clone()
            z_arr = coords_seq[0][:, 2:3].clone()

            # Temperature sequence (N_t,)
            T_vals = torch.tensor(T_seq, dtype=torch.float32, device=device)
            Th_vals = (T_vals - self.Temp_min) / self.Temp_range   # (N_t,)

            # Initial LSTM hidden state (zero)
            h_state = torch.zeros(
                self.model.lstm_layers, N_pts, self.model.lstm_hidden,
                device=device, dtype=torch.float32)
            c_state = torch.zeros_like(h_state)

            rf_indices_torch = torch.tensor(rf_indices, dtype=torch.long, device=device)

            # NOTE: We run the LSTM step-by-step (forward_step) rather than
            # forward_sequence + per-step autograd.grad.
            # The original design called autograd.grad(u_seq[:,n,:], x_seq_raw) inside
            # a loop over N_t, requiring full BPTT through ALL N_t LSTM steps per call.
            # That is O(N_t²) backward passes — the source of the 24-hour hang.
            #
            # Per-step forward with DETACHED hidden state gives O(1) BPTT for strain:
            #   u_n depends only on the CURRENT step's spatial encoder + one LSTM cell,
            #   so autograd.grad(u_n, x_n) traverses exactly ONE LSTM step.
            # Hidden-state BPTT for data/RF losses is kept intact within each step
            # (create_graph=True); only the cross-step hidden dependency is truncated.
            # Shift-parameter gradients flow through the q recursion (not the LSTM),
            # so they are unaffected by hidden-state truncation.

            # ---- Accumulate losses ----
            loss_data_total     = torch.zeros((), device=device)
            loss_strain_total   = torch.zeros((), device=device)
            loss_rf_total       = torch.zeros((), device=device)
            loss_bc_left_total  = torch.zeros((), device=device)
            loss_bc_right_total = torch.zeros((), device=device)
            loss_traction_total = torch.zeros((), device=device)

            tol = 1e-4
            N_prony = self.mat.N_prony
            rho = self.mat.rho.view(1, 1, N_prony)
            q_prev = torch.zeros(N_pts, 6, N_prony, device=device, dtype=torch.float32)
            truncate_every = 50

            for n in range(N_t):
                u_data_n = u_data_seq[n]

                # Fresh leaf tensors per step → autograd.grad(u_n, x_n) traverses
                # only the CURRENT LSTM step (not the full sequence).
                x_n = x_arr.clone().requires_grad_(True)   # (N_pts, 1)
                y_n = y_arr.clone().requires_grad_(True)
                z_n = z_arr.clone().requires_grad_(True)
                xh_n = (x_n - self.x_min) / self.Lx
                yh_n = (y_n - self.y_min) / self.Ly
                zh_n = (z_n - self.z_min) / self.Lz
                Th_n = Th_vals[n].view(1, 1).expand(N_pts, 1)   # (N_pts, 1)

                with torch.backends.cudnn.flags(enabled=False):
                    u_pred_n, (h_state, c_state) = self.model.forward_step(
                        xh_n, yh_n, zh_n, Th_n,
                        h_state.detach(), c_state.detach(),
                    )
                # h_state.detach(): truncate hidden-state BPTT to avoid O(N_t²) cost.
                # Shift-parameter gradients flow through q_n recursion, not LSTM hidden
                # state, so truncation does not impair parameter identification.

                strain_n = self._compute_strain_from_step(u_pred_n, x_n, y_n, z_n)
                strain_mat_n = self.to_material_coords(strain_n)

                if n == 0:
                    q_n = q_prev
                else:
                    dt = dt_seq[n - 1]
                    T_n = torch.full((N_pts, 1), T_seq[n], dtype=torch.float32, device=device)

                    aT_n = self.mat.shift_factor(T_n)
                    aT_n = torch.nan_to_num(aT_n, nan=1.0, posinf=1e12, neginf=1e-8)
                    aT_n = torch.clamp(aT_n, min=1e-8, max=1e12)

                    delta_xi = (dt / aT_n).unsqueeze(-1)   # (N_pts,1,1)
                    delta_xi = torch.nan_to_num(delta_xi, nan=0.0, posinf=1e6, neginf=0.0)
                    dxi_over_rho = torch.clamp(delta_xi / rho, min=0.0, max=80.0)
                    exp_factor = torch.exp(-dxi_over_rho)
                    relax_factor = 1.0 - exp_factor

                    q_base = q_prev.detach() if (n % truncate_every == 0) else q_prev
                    strain_safe = torch.nan_to_num(
                        strain_mat_n, nan=0.0, posinf=1e6, neginf=-1e6
                    )
                    q_n = exp_factor * q_base + strain_safe.unsqueeze(-1) * relax_factor
                    q_n = torch.nan_to_num(q_n, nan=0.0, posinf=1e6, neginf=-1e6)

                stress_n   = self.compute_stress(strain_mat_n, q_n)
                q_prev = q_n

                # Data loss
                loss_data_total += torch.mean((u_pred_n - u_data_n) ** 2) / self.u_ref_sq

                # Strain loss
                if strain_data_seq[n] is not None:
                    strain_fe = strain_data_seq[n]
                    eps_ref_sq = 0.15 ** 2
                    loss_strain_total += torch.mean(
                        (strain_n - strain_fe) ** 2
                    ) / eps_ref_sq

                # RF loss (completely skipped when lambda_rf == 0, e.g. no-RF runs)
                if self.lambda_rf > 0.0:
                    rf_strain_mat = strain_mat_n[rf_indices_torch, :]
                    rf_q_n = q_n[rf_indices_torch, :, :]
                    stress_rf = self.compute_stress(rf_strain_mat, rf_q_n)
                    rf_pred = self.compute_reaction_force(stress_rf)
                    t_val = sampled_times[n]
                    # RF is physically meaningful only while right end is constrained.
                    if t_val < 100.0:
                        rf_target = torch.tensor(
                            self.fe_loader.get_rf_by_time(t_val),
                            dtype=torch.float32, device=device
                        )
                        rf_denom = torch.clamp(torch.abs(rf_target), min=self.rf_rel_floor)
                        loss_rf_total += ((rf_pred - rf_target) / rf_denom) ** 2

                # BC right
                u_bc_r_pred = u_pred_n[rf_indices_torch, :]
                u_bc_r_tgt  = u_data_seq[n][rf_indices_torch, :]
                loss_bc_right_total += torch.mean(
                    (u_bc_r_pred - u_bc_r_tgt) ** 2
                ) / self.u_ref_sq

                # BC left (u=0)
                if len(left_indices) > 0:
                    li = torch.tensor(left_indices, dtype=torch.long, device=device)
                    loss_bc_left_total += torch.mean(
                        u_pred_n[li, :] ** 2
                    ) / self.u_ref_sq

                # Traction-free lateral
                if len(yz_face_indices) > 0:
                    yi = torch.tensor(yz_face_indices, dtype=torch.long, device=device)
                    coords_yz = coords_seq[n][yi, :]
                    stress_yz = stress_n[yi, :]
                    sigma_ref_sq = 100.0 ** 2
                    y_min = self.bounds['y_min']; y_max = self.bounds['y_max']
                    z_min = self.bounds['z_min']; z_max = self.bounds['z_max']
                    is_y = (torch.abs(coords_yz[:, 1] - y_min) < tol) | \
                           (torch.abs(coords_yz[:, 1] - y_max) < tol)
                    is_z = (torch.abs(coords_yz[:, 2] - z_min) < tol) | \
                           (torch.abs(coords_yz[:, 2] - z_max) < tol)
                    if is_y.any():
                        sy = stress_yz[is_y]
                        ty = torch.stack([sy[:, 3], sy[:, 1], sy[:, 5]], dim=1)
                        loss_traction_total += torch.mean(ty ** 2) / sigma_ref_sq
                    if is_z.any():
                        sz = stress_yz[is_z]
                        tz = torch.stack([sz[:, 4], sz[:, 5], sz[:, 2]], dim=1)
                        loss_traction_total += torch.mean(tz ** 2) / sigma_ref_sq

            # Average
            loss_data     = loss_data_total     / N_t
            loss_strain   = loss_strain_total   / N_t
            loss_rf       = loss_rf_total       / N_t
            loss_bc_left  = loss_bc_left_total  / N_t
            loss_bc_right = loss_bc_right_total / N_t
            loss_traction = loss_traction_total / N_t
            loss_penalty  = self.mat.compute_parameter_penalty()

            loss_total = (
                self.lambda_data     * loss_data +
                self.lambda_strain   * loss_strain +
                self.lambda_bc_left  * loss_bc_left +
                self.lambda_bc_right * loss_bc_right +
                self.lambda_traction * loss_traction +
                self.lambda_rf       * loss_rf +
                50.0                 * loss_penalty
            )

            loss_total.backward()

            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            torch.nn.utils.clip_grad_norm_(shift_params, max_norm=0.5)

            optimizer.step()
            scheduler.step()

            self.loss_history.append(loss_total.item())
            self.loss_components_history.append({
                'data':     loss_data.item(),
                'strain':   loss_strain.item(),
                'bc_left':  loss_bc_left.item(),
                'bc_right': loss_bc_right.item(),
                'traction': loss_traction.item(),
                'rf':       loss_rf.item(),
                'pde':      0.0,   # LSTM version skips PDE for memory
                'penalty':  loss_penalty.item(),
            })
            self.param_history.append({
                'C1':   self.mat.C1.item(),
                'C2':   self.mat.C2.item(),
                'Ea_R': self.mat.Ea_R.item(),
            })

            if epoch % log_interval == 0 or epoch == epochs - 1:
                lr_net = optimizer.param_groups[0]['lr']
                lr_sft = optimizer.param_groups[1]['lr']
                print(
                    f"[Epoch {epoch:5d}] total={loss_total.item():.4e} | "
                    f"data={loss_data.item():.4e} strain={loss_strain.item():.4e} "
                    + (f"rf={loss_rf.item():.4e}" if self.lambda_rf > 0.0 else "rf=0.0000e+00")
                )
                print(
                    f"  [Shift] C1={self.mat.C1.item():.3f}(T:{self.mat.TRUE_C1})  "
                    f"C2={self.mat.C2.item():.2f}(T:{self.mat.TRUE_C2})  "
                    f"Ea_R={self.mat.Ea_R.item():.0f}(T:{self.mat.TRUE_Ea_R})"
                )
                print(f"  [LR] net={lr_net:.2e} shift={lr_sft:.2e}  N_t={N_t}")

        print(f"\nLSTM training complete: {time.time() - start_time:.1f}s")


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------


## Training Configuration


In [ ]:
# RF loss control
use_rf_loss = True
lambda_rf = 5.0 if use_rf_loss else 0.0

# Remaining weights (aligned with EX4 setting)
lambda_data = 5.0
lambda_strain = 2.0
lambda_pde = 0.0
lambda_bc_left = 1.0
lambda_bc_right = 5.0
lambda_traction = 5.0

# Training hyperparameters
epochs = 10000
batch_size = 48
seq_length = 24
lr_network = 1e-3
lr_params = 1e-3
stride = 5

set_global_seed(42)
print(f'use_rf_loss={use_rf_loss}, lambda_rf={lambda_rf}')


## Training and Saving Results


In [ ]:
script_dir = Path('.')
data_dir = script_dir / 'EX-4-RESULTS'
rf_file = script_dir / 'ex-4-NonIsothermalSMCycle-RF.csv'
u_file = script_dir / 'ex-4-NonIsothermalSMCycle-U.csv'
ft_file = script_dir / 'frame-time.csv'

fe_loader = FEDataLoader(data_dir, rf_file, u_file, ft_file, stride=stride)
fe_loader.load_all_data()
bounds = fe_loader.get_domain_bounds()

mat_params = InverseShiftParams().to(device)
model = LSTMInverseEX4(
    spatial_hidden=(64, 64),
    lstm_hidden=64,
    lstm_layers=2,
    dropout=0.0,
).to(device)

solver = InverseLSTMSolverEX4(
    model, mat_params, fe_loader, bounds,
    lambda_data=lambda_data,
    lambda_strain=lambda_strain,
    lambda_pde=lambda_pde,
    lambda_bc_left=lambda_bc_left,
    lambda_bc_right=lambda_bc_right,
    lambda_traction=lambda_traction,
    lambda_rf=lambda_rf,
)

solver.train(
    epochs=epochs,
    batch_size=batch_size,
    lr_network=lr_network,
    lr_params=lr_params,
    log_interval=100,
    seq_length=seq_length,
)

tag = 'rf' if lambda_rf > 0 else 'norf'
loss_csv = script_dir / f'nb_ex4_lstm_{tag}_loss_history.csv'
param_csv = script_dir / f'nb_ex4_lstm_{tag}_param_history.csv'
model_path = script_dir / f'nb_ex4_lstm_{tag}_model.pth'
plot_path = script_dir / f'nb_ex4_lstm_{tag}_training_history.png'

solver.save_loss_history(loss_csv)
solver.save_param_history(param_csv)
plot_training_history(solver, script_dir)

src = script_dir / 'ex4_training_history.png'
if src.exists():
    import shutil
    shutil.copy2(src, plot_path)

torch.save({
    'model_state_dict': model.state_dict(),
    'mat_params_state_dict': mat_params.state_dict(),
    'final_params': {
        'C1': mat_params.C1.item(),
        'C2': mat_params.C2.item(),
        'Ea_R': mat_params.Ea_R.item(),
    },
    'lambda_rf': float(lambda_rf),
}, model_path)

print('Saved:')
print(' ', loss_csv)
print(' ', param_csv)
print(' ', model_path)
print(' ', plot_path)
